<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2_4_Support_Vector_Machines_(SVM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part II. Classification. Support Vector Machines (SVM)

# Глава 6. Метод опорных векторов (SVM). Часть I

Логистическая регрессия строит вероятностную разделяющую поверхность, но не предъявляет жёстких требований к ширине разделяющей полосы. Метод опорных векторов, напротив, ищет гиперплоскость, которая не просто разделяет классы, а отстоит от ближайших точек обоих классов на максимально возможное расстояние. В этой главе мы рассмотрим линейный SVM для строго разделимых данных — так называемый **hard‑margin SVM**, — заложивший геометрический и оптимизационный фундамент всего семейства опорных методов.

## 1. Линейная разделимость и зазор

Рассматривается задача бинарной классификации. Обучающая выборка состоит из $n$ пар $(\mathbf{x}_i, y_i)$, где $\mathbf{x}_i \in \mathbb{R}^D$, а метки классов принимают значения
$$
y_i \in \{-1, +1\}.
$$
Выбор именно $\pm 1$ продиктован удобством: знак метки будет непосредственно указывать, по какую сторону от разделяющей гиперплоскости должен находиться объект.

Разделяющая гиперплоскость задаётся уравнением
$$
f(\mathbf{x}) = \mathbf{w}^T\mathbf{x} + b = 0,
$$
где $\mathbf{w} \in \mathbb{R}^D$ — вектор нормали, $b \in \mathbb{R}$ — свободный член. Решающее правило принимает вид
$$
\hat{y} = \operatorname{sign}\!\bigl(f(\mathbf{x})\bigr).
$$
Если $f(\mathbf{x}) > 0$, объект относится к классу $+1$, иначе — к классу $-1$. Предполагается, что данные **линейно разделимы**, то есть существует хотя бы одна гиперплоскость, безошибочно классифицирующая все обучающие примеры.

Введём понятие **функционального зазора** (functional margin) для одного наблюдения:
$$
\gamma_i = y_i\bigl(\mathbf{w}^T\mathbf{x}_i + b\bigr).
$$
Если классификация верна, то знаки $y_i$ и $\mathbf{w}^T\mathbf{x}_i + b$ совпадают, и $\gamma_i > 0$. Чем больше $\gamma_i$, тем «увереннее» объект лежит в своей полуплоскости. Функциональный зазор всей выборки определяется как $\min_i \gamma_i$.

У функционального зазора есть принципиальный недостаток: он зависит от масштаба вектора $(\mathbf{w}, b)$. Умножив $\mathbf{w}$ и $b$ на одно и то же положительное число, мы не изменим гиперплоскость и решающее правило, но функциональный зазор возрастёт во столько же раз. Следовательно, максимизировать функциональный зазор напрямую нельзя — это тривиально приводит к неограниченному росту нормы.

Истинной геометрической мерой является **геометрический зазор** (geometric margin). Вспомним, что расстояние от точки $\mathbf{x}_i$ до гиперплоскости $\mathbf{w}^T\mathbf{x} + b = 0$ равно
$$
\frac{|\mathbf{w}^T\mathbf{x}_i + b|}{\|\mathbf{w}\|}.
$$
Поскольку для правильно классифицированного объекта $y_i$ и выражение в числителе имеют одинаковый знак, расстояние до гиперплоскости с учётом класса записывается как
$$
\frac{y_i(\mathbf{w}^T\mathbf{x}_i + b)}{\|\mathbf{w}\|} = \frac{\gamma_i}{\|\mathbf{w}\|}.
$$
Геометрический зазор всей выборки — это минимальное из таких расстояний:
$$
\rho = \min_{i=1,\dots,n} \frac{y_i(\mathbf{w}^T\mathbf{x}_i + b)}{\|\mathbf{w}\|}.
$$
Величина $\rho$ имеет наглядный геометрический смысл: она равна половине ширины полосы, свободной от точек данных и разделяющей классы.


Запустите ячейку ниже, чтобы увидеть разделяющую гиперплоскость, границы зазора и опорные векторы. Обратите внимание на геометрический зазор ρ, выводимый после обучения.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm

# ================================================
# 1. Исходные данные: два линейно разделимых класса
# ================================================
np.random.seed(42)
X = np.array([[1.0, 2.0], [2.0, 3.0], [3.0, 3.0], [2.0, 1.0], [3.0, 2.0],
              [5.0, 5.0], [6.0, 6.0], [7.0, 5.0], [5.0, 6.0], [6.0, 5.0]])
y = np.array([-1, -1, -1, -1, -1, 1, 1, 1, 1, 1])

# ================================================
# 2. Обучение hard‑margin SVM (C очень большое)
# ================================================
clf = svm.SVC(kernel='linear', C=1e5)
clf.fit(X, y)

# Параметры гиперплоскости: w и b
w = clf.coef_[0]               # вектор нормали
b = clf.intercept_[0]          # свободный член

# Геометрический зазор: ρ = 1 / ||w||
margin = 1 / np.linalg.norm(w)

print(f'Вектор нормали w = {np.round(w, 3)}')
print(f'Свободный член b = {b:.3f}')
print(f'Геометрический зазор ρ = 1 / ||w|| = {margin:.3f}')

# ================================================
# 3. Построение графика
# ================================================
plt.figure(figsize=(8, 6))

# Все точки
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', s=80, edgecolors='k',
            label='Обучающие примеры')

# Опорные векторы (обведены кружком)
sv = clf.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=200, facecolors='none',
            edgecolors='k', linewidths=2, label='Опорные векторы')

# Сетка для рисования линий уровня
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()
xx = np.linspace(xlim[0], xlim[1], 50)
yy = np.linspace(ylim[0], ylim[1], 50)
XX, YY = np.meshgrid(xx, yy)
Z = clf.decision_function(np.c_[XX.ravel(), YY.ravel()])
Z = Z.reshape(XX.shape)

# Линии: границы зазора (w·x+b = ±1) и сама гиперплоскость (0)
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1],
           linestyles=['--', '-', '--'], alpha=0.7,
           labels=['Граница зазора (-1)', 'Гиперплоскость', 'Граница зазора (+1)'])

# Аннотации
plt.title(f'Hard‑margin SVM\nГеометрический зазор = {margin:.3f}')
plt.xlabel('Признак 1')
plt.ylabel('Признак 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Идея метода опорных векторов** — среди всех гиперплоскостей, безошибочно разделяющих классы, выбрать ту, для которой зазор $\rho$ максимален. Интуитивно: чем шире свободная полоса, тем менее чувствительной будет классификация к малым возмущениям данных и тем выше обобщающая способность модели.

## 2. Прямая задача hard‑margin SVM

Максимизация $\rho$ непосредственно неудобна. Однако мы можем воспользоваться тем, что масштаб $(\mathbf{w}, b)$ не влияет на геометрический зазор. Без ограничения общности можно отмасштабировать параметры так, чтобы для ближайших к гиперплоскости точек выполнялось
$$
y_i(\mathbf{w}^T\mathbf{x}_i + b) = 1.
$$
Действительно, пусть $(\mathbf{w}', b')$ — произвольная разделяющая гиперплоскость. Положим $\gamma' = \min_i y_i(\mathbf{w}'^T\mathbf{x}_i + b') > 0$. Тогда пара $(\mathbf{w}, b) = (\mathbf{w}'/\gamma', b'/\gamma')$ даёт ту же самую гиперплоскость, но с минимальным функциональным зазором, равным единице. Никакая информация о направлении или положении гиперплоскости при этом не теряется.

После такой нормировки функциональный зазор любого объекта не меньше единицы:
$$
y_i(\mathbf{w}^T\mathbf{x}_i + b) \ge 1, \qquad i=1,\dots,n.
$$
Геометрический зазор для гиперплоскости, удовлетворяющей этим ограничениям, становится равным
$$
\rho = \frac{1}{\|\mathbf{w}\|}.
$$
Таким образом, максимизация зазора эквивалентна минимизации $\|\mathbf{w}\|$. Для удобства оптимизации (гладкость и выпуклость) минимизируют квадрат нормы с коэффициентом $1/2$. Прямая задача **hard‑margin SVM** записывается следующим образом:
$$
\boxed{\begin{aligned}
\min_{\mathbf{w}, b} \quad & \frac{1}{2}\|\mathbf{w}\|^2 \\
\text{при ограничениях} \quad & y_i(\mathbf{w}^T\mathbf{x}_i + b) \ge 1, \quad i = 1,\dots,n.
\end{aligned}}
\tag{2.1}
$$
Это задача выпуклого квадратичного программирования: целевая функция — выпуклый квадратичный функционал, ограничения — линейные неравенства. Выпуклость гарантирует, что любой локальный минимум является глобальным, а также открывает возможность применять эффективные численные методы и строить двойственную задачу.

> **Углублённый взгляд: единственность решения**  
> Целевая функция $\frac{1}{2}\|\mathbf{w}\|^2$ является строго выпуклой по $\mathbf{w}$. Допустимое множество $\mathcal{F} = \{(\mathbf{w},b) \mid y_i(\mathbf{w}^T\mathbf{x}_i+b) \ge 1,\; \forall i\}$ — пересечение замкнутых полупространств, следовательно, замкнутое выпуклое множество. Задача минимизации строго выпуклой функции на выпуклом множестве не может иметь двух различных решений с разными $\mathbf{w}$: если $(\mathbf{w}_1,b_1)$ и $(\mathbf{w}_2,b_2)$ оптимальны, то по строгой выпуклости для любого $\lambda \in (0,1)$ значение функции в средней точке было бы строго меньше, что противоречит оптимальности. Поэтому **вектор $\mathbf{w}^*$ единственен**.  
> Смещение $b^*$ может быть неединственным в вырожденных ситуациях, когда несколько опорных векторов каждого класса лежат строго на границах зазора и их расположение допускает сдвиг гиперплоскости без нарушения ограничений. В практически важных случаях (например, при наличии в каждом классе хотя бы одного опорного вектора, не лежащего с другими на одной прямой) $b^*$ также единственно.

## 3. Лагранжиан и переход к двойственной задаче

Для решения задачи (2.1) воспользуемся методом множителей Лагранжа. Введём двойственные переменные $\alpha_i \ge 0$, $i=1,\dots,n$, и построим лагранжиан:
$$
L(\mathbf{w}, b, \boldsymbol{\alpha}) = \frac{1}{2}\|\mathbf{w}\|^2 - \sum_{i=1}^{n} \alpha_i \bigl(y_i(\mathbf{w}^T\mathbf{x}_i + b) - 1\bigr).
\tag{3.1}
$$
Знак «минус» перед суммой выбран для удобства: ограничения записаны в виде $g_i(\mathbf{w},b) \le 0$, где $g_i = 1 - y_i(\mathbf{w}^T\mathbf{x}_i + b) \le 0$. Двойственная функция определяется как инфимум лагранжиана по прямым переменным:
$$
g(\boldsymbol{\alpha}) = \inf_{\mathbf{w}, b} L(\mathbf{w}, b, \boldsymbol{\alpha}).
$$

Найдём точку минимума аналитически. Приравняем нулю градиенты:
$$
\frac{\partial L}{\partial \mathbf{w}} = \mathbf{w} - \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i = \mathbf{0}
\quad\Longrightarrow\quad
\mathbf{w} = \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i,
\tag{3.2}
$$
$$
\frac{\partial L}{\partial b} = -\sum_{i=1}^{n} \alpha_i y_i = 0
\quad\Longrightarrow\quad
\sum_{i=1}^{n} \alpha_i y_i = 0.
\tag{3.3}
$$

Подставим (3.2) и (3.3) обратно в $L$. Заметим, что
$$
\frac{1}{2}\|\mathbf{w}\|^2 = \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j),
$$
а также
$$
\sum_{i} \alpha_i y_i \mathbf{w}^T \mathbf{x}_i = \sum_{i,j} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j).
$$
С учётом $\sum_i \alpha_i y_i = 0$ выражение $\sum_i \alpha_i y_i b$ обращается в ноль. Окончательно:
$$
\begin{aligned}
L(\mathbf{w}, b, \boldsymbol{\alpha})
&= \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j) - \sum_{i,j} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j) + \sum_{i=1}^{n} \alpha_i \\
&= \sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i=1}^{n}\sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j).
\end{aligned}
$$

Таким образом, двойственная задача заключается в максимизации $g(\boldsymbol{\alpha})$ при ограничениях $\alpha_i \ge 0$ и $\sum_i \alpha_i y_i = 0$:
$$
\boxed{\begin{aligned}
\max_{\boldsymbol{\alpha}} \quad & \sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i=1}^{n}\sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j) \\
\text{при ограничениях} \quad & \alpha_i \ge 0, \quad i = 1,\dots,n, \\
& \sum_{i=1}^{n} \alpha_i y_i = 0.
\end{aligned}}
\tag{3.4}
$$

Двойственная задача обладает двумя замечательными свойствами. Во‑первых, она зависит только от скалярных произведений $\mathbf{x}_i^T \mathbf{x}_j$, а не от самих векторов признаков. Во‑вторых, число переменных равно числу наблюдений $n$, а не размерности $D$ (прямая переменная $\mathbf{w}$ исчезла). Первое свойство открывает путь к **ядерному трюку** (kernel trick), позволяющему без изменения формализма строить нелинейные разделяющие поверхности.

## 4. Условия Каруша–Куна–Таккера (KKT) и опорные векторы

Поскольку прямая задача (2.1) выпуклая, а все ограничения аффинны, условия Каруша–Куна–Таккера являются необходимыми и достаточными для оптимальности. Для точки $(\mathbf{w}^*, b^*, \boldsymbol{\alpha}^*)$ они имеют вид:

1. **Стационарность лагранжиана:**
   $$
   \mathbf{w}^* = \sum_{i=1}^{n} \alpha_i^* y_i \mathbf{x}_i, \qquad
   \sum_{i=1}^{n} \alpha_i^* y_i = 0.
   $$
2. **Прямая допустимость:**
   $$
   y_i\bigl((\mathbf{w}^*)^T\mathbf{x}_i + b^*\bigr) \ge 1, \quad \forall i.
   $$
3. **Двойственная допустимость:**
   $$
   \alpha_i^* \ge 0, \quad \forall i.
   $$
4. **Условие дополняющей нежёсткости (complementary slackness):**
   $$
   \alpha_i^* \Bigl( y_i\bigl((\mathbf{w}^*)^T\mathbf{x}_i + b^*\bigr) - 1 \Bigr) = 0, \quad \forall i.
   \tag{4.1}
   $$

Условие дополняющей нежёсткости раскрывает структуру решения. Для каждого наблюдения возможно ровно одно из двух:

- **$\alpha_i^* > 0$.** Тогда скобка в (4.1) обязана равняться нулю, откуда
  $$
  y_i\bigl((\mathbf{w}^*)^T\mathbf{x}_i + b^*\bigr) = 1.
  $$
  Такие точки лежат точно на границе зазора и называются **опорными векторами** (support vectors).
- **$\alpha_i^* = 0$.** В этом случае ограничение $y_i(\mathbf{w}^T\mathbf{x}_i + b) \ge 1$ выполняется со строгим неравенством. Точка находится внутри своей полуплоскости, дальше границы зазора, и её множитель Лагранжа равен нулю — она **не влияет** на решение.

Это наблюдение объясняет название метода: гиперплоскость «опирается» лишь на небольшую часть обучающих объектов — опорные векторы. Формально, из (3.2) видно, что вектор $\mathbf{w}^*$ является линейной комбинацией только тех $\mathbf{x}_i$, для которых $\alpha_i^* > 0$. Остальные точки можно удалить из выборки, и решение не изменится. **Разреженность** по опорным векторам — одно из ключевых вычислительных и концептуальных преимуществ SVM.


Проверим это на практике. Ячейка ниже обучает SVM только на опорных векторах, выделенных на предыдущем графике, и показывает, что граница не изменилась.




In [ ]:
# ================================================
# Демонстрация разреженности: удалим неопорные векторы
# ================================================
# Индексы опорных векторов (из предыдущей модели)
support_indices = clf.support_

# Оставляем только опорные векторы
X_sv = X[support_indices]
y_sv = y[support_indices]

# Обучаем SVM только на опорных векторах
clf_sv = svm.SVC(kernel='linear', C=1e5)
clf_sv.fit(X_sv, y_sv)

w_sv = clf_sv.coef_[0]
b_sv = clf_sv.intercept_[0]

print('Параметры после удаления неопорных векторов:')
print(f'w = {np.round(w_sv, 3)}')
print(f'b = {b_sv:.3f}')
print('Вектор w совпадает с исходным?', np.allclose(w, w_sv))

# Визуализация
plt.figure(figsize=(8, 6))
# Все исходные точки (серые, неиспользуемые)
plt.scatter(X[:, 0], X[:, 1], c='lightgray', s=60, edgecolors='gray',
            label='Удалённые (неопорные)')
# Только опорные векторы
plt.scatter(X_sv[:, 0], X_sv[:, 1], c=y_sv, cmap='bwr', s=80,
            edgecolors='k', label='Опорные векторы')

# Граница, построенная только по опорным векторам
xx = np.linspace(plt.xlim()[0], plt.xlim()[1], 50)
yy = np.linspace(plt.ylim()[0], plt.ylim()[1], 50)
XX, YY = np.meshgrid(xx, yy)
Z_sv = clf_sv.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
plt.contour(XX, YY, Z_sv, colors='k', levels=[-1, 0, 1],
            linestyles=['--', '-', '--'])
plt.title('Решение SVM только по опорным векторам (граница не изменилась)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


> **Углублённый взгляд: сильная двойственность и квалификация Слейтера**  
> Для задачи выпуклой оптимизации с аффинными ограничениями сильная двойственность (равенство оптимальных значений прямой и двойственной задач) гарантируется, если существует точка, строго удовлетворяющая всем неравенствам — так называемое **условие Слейтера**. В случае hard‑margin SVM требуется найти $(\tilde{\mathbf{w}}, \tilde{b})$, для которой  
> $$
> y_i(\tilde{\mathbf{w}}^T\mathbf{x}_i + \tilde{b}) > 1, \quad \forall i.
> $$
> Покажем, что линейная разделимость гарантирует выполнение этого условия.  
> Поскольку классы линейно разделимы, существует гиперплоскость с параметрами $(\mathbf{w}', b')$, для которой $y_i(\mathbf{w}'^T\mathbf{x}_i + b') > 0$ при всех $i$. Определим минимальный функциональный зазор этой гиперплоскости:  
> $$
> \gamma_{\min} = \min_{i} y_i(\mathbf{w}'^T\mathbf{x}_i + b') > 0.
> $$
> Теперь возьмём масштабированные параметры  
> $$
> \tilde{\mathbf{w}} = \frac{2}{\gamma_{\min}}\mathbf{w}', \qquad \tilde{b} = \frac{2}{\gamma_{\min}}b'.
> $$
> Тогда для любого объекта $i$ выполняется  
> $$
> y_i(\tilde{\mathbf{w}}^T\mathbf{x}_i + \tilde{b}) = \frac{2}{\gamma_{\min}}\, y_i(\mathbf{w}'^T\mathbf{x}_i + b') \ge \frac{2}{\gamma_{\min}}\cdot \gamma_{\min} = 2 > 1.
> $$
> Таким образом, точка $(\tilde{\mathbf{w}}, \tilde{b})$ является внутренней точкой допустимой области, условие Слейтера выполнено, и разрыв двойственности равен нулю — оптимальные значения прямой и двойственной задач совпадают. Это фундаментальный факт, позволяющий перейти к двойственной задаче без потери точности.

In [ ]:
# Возьмём уже обученную гиперплоскость (из первой модели)
w_orig = clf.coef_[0]
b_orig = clf.intercept_[0]

# Масштабируем в 2 раза, как в условии Слейтера
w_scaled = 2 * w_orig
b_scaled = 2 * b_orig

# Проверим, что знак предсказаний остался прежним
pred_orig = np.sign(np.dot(X, w_orig) + b_orig)
pred_scaled = np.sign(np.dot(X, w_scaled) + b_scaled)

print("Предсказания совпадают?", np.array_equal(pred_orig, pred_scaled))

# Функциональный зазор после масштабирования увеличился в 2 раза
margin_func_orig = np.min(y * (np.dot(X, w_orig) + b_orig))
margin_func_scaled = np.min(y * (np.dot(X, w_scaled) + b_scaled))
print(f"Исходный функциональный зазор: {margin_func_orig:.3f}")
print(f"После масштабирования: {margin_func_scaled:.3f}")
print("Во сколько раз увеличился:", margin_func_scaled / margin_func_orig)


## 5. Восстановление решающего правила

После того как двойственная задача (3.4) решена и найдены оптимальные множители $\alpha_i^*$, вектор $\mathbf{w}^*$ немедленно восстанавливается по формуле (3.2):
$$
\mathbf{w}^* = \sum_{i=1}^{n} \alpha_i^* y_i \mathbf{x}_i.
$$
Для вычисления смещения $b^*$ воспользуемся тем, что для любого опорного вектора (индекса $s$, для которого $\alpha_s^* > 0$) выполняется равенство
$$
y_s\bigl((\mathbf{w}^*)^T\mathbf{x}_s + b^*\bigr) = 1.
$$
Так как $y_s \in \{-1, +1\}$, умножив обе части на $y_s$, получаем
$$
(\mathbf{w}^*)^T\mathbf{x}_s + b^* = y_s,
$$
откуда
$$
b^* = y_s - (\mathbf{w}^*)^T\mathbf{x}_s = y_s - \sum_{i=1}^{n} \alpha_i^* y_i (\mathbf{x}_i^T \mathbf{x}_s).
\tag{5.1}
$$
Из‑за возможной численной погрешности на практике значение $b^*$ усредняют по всем опорным векторам:
$$
b^* = \frac{1}{|S|} \sum_{s \in S} \left( y_s - \sum_{i=1}^{n} \alpha_i^* y_i (\mathbf{x}_i^T \mathbf{x}_s) \right),
$$
где $S = \{ i \mid \alpha_i^* > 0 \}$ — множество индексов опорных векторов. Это даёт более устойчивую оценку.

Итоговое решающее правило для нового объекта $\mathbf{x}$ записывается без явного обращения к вектору $\mathbf{w}^*$, через двойственные переменные и скалярные произведения:
$$
\boxed{f(\mathbf{x}) = \sum_{i=1}^{n} \alpha_i^* y_i (\mathbf{x}_i^T \mathbf{x}) + b^*, \qquad
\hat{y} = \operatorname{sign}\!\bigl(f(\mathbf{x})\bigr).}
\tag{5.2}
$$
В этом представлении участвуют только опорные векторы (остальные имеют $\alpha_i^* = 0$), что делает классификацию эффективной и открывает возможность бесконечномерных обобщений с помощью ядер.

Таким образом, hard‑margin SVM даёт геометрически мотивированный, выпуклый и разреженный линейный классификатор, максимизирующий зазор. В следующей части главы мы откажемся от жёсткой разделимости и введём мягкий зазор (soft‑margin SVM), позволяющий работать с зашумлёнными и нелинейно разделимыми данными.



In [ ]:
# @title Проверка восстановления b через опорные векторы
# Найдём опорные векторы (уже есть clf)
sv = clf.support_vectors_
alpha_sv = clf.dual_coef_[0]   # y_i * alpha_i для опорных векторов
y_sv = y[clf.support_]

# Восстановим w по опорным векторам
w_manual = np.sum(alpha_sv[:, None] * sv, axis=0)

# Вычислим b по каждому опорному вектору и усредним
b_manual = np.mean([y_sv[k] - np.dot(w_manual, sv[k]) for k in range(len(sv))])

print(f'b (sklearn):    {clf.intercept_[0]:.6f}')
print(f'b (восст. и усредн.): {b_manual:.6f}')
print('Совпадают:', np.isclose(clf.intercept_[0], b_manual))



## Вопросы для самопроверки

### Для начинающих

1. Дайте определение геометрического зазора классификации. Как он связан с расстоянием от точки до гиперплоскости?
2. Почему максимизация зазора в hard‑margin SVM сводится к минимизации $\frac{1}{2}\|\mathbf{w}\|^2$ при ограничениях $y_i(\mathbf{w}^T\mathbf{x}_i + b) \ge 1$?
3. Запишите прямую задачу оптимизации для линейного hard‑margin SVM.
4. Для чего вводятся множители Лагранжа $\alpha_i$ и как выглядит двойственная задача SVM?
5. Какие точки обучающей выборки называются опорными векторами? Каким условиям KKT они удовлетворяют?
6. Как вычислить смещение $b$ после нахождения оптимальных множителей $\boldsymbol{\alpha}^*$?

### Для профессионалов

1. Докажите, что решение $\mathbf{w}^*$ в задаче hard‑margin SVM единственно. В каких случаях может быть неединственным $b^*$ и почему?
2. Проведите полный вывод двойственной задачи SVM, начиная с лагранжиана и заканчивая целевой функцией $g(\boldsymbol{\alpha})$. Покажите, куда исчезает зависимость от $b$.
3. Проанализируйте условия Каруша–Куна–Таккера для SVM и покажите, как из условия дополняющей нежёсткости следует, что решение $\mathbf{w}^*$ выражается только через опорные векторы (разреженность).
4. Проверьте выполнение условия Слейтера для задачи hard‑margin SVM и объясните, почему в этом случае гарантирована сильная двойственность.
5. Почему двойственная задача зависит только от скалярных произведений $\mathbf{x}_i^T\mathbf{x}_j$? Какое принципиальное значение это имеет для перехода к нелинейным классификаторам?



## 6. Потребность в мягком зазоре (soft‑margin)

Предположение о строгой линейной разделимости классов, на котором строился hard‑margin SVM, на практике выполняется крайне редко. Реальные данные обычно содержат шум, выбросы или просто не допускают безошибочного разделения гиперплоскостью. В таких условиях задача hard‑margin SVM либо не имеет ни одного допустимого решения, либо выделяет гиперплоскость, определяемую единственной «неудобной» точкой, что приводит к узкому зазору и плохому обобщению.

Чтобы сделать метод работоспособным на любых данных, необходимо ослабить жёсткое требование безошибочной классификации. Идея состоит в том, чтобы разрешить некоторым объектам нарушать границу зазора, но назначить **штраф** за каждое такое нарушение, пропорциональный его величине. Для этого вводятся неотрицательные **slack‑переменные** (переменные нежёсткости) $\xi_i \ge 0$, по одной на каждое наблюдение. Ограничения заменяются на
$$
y_i(\mathbf{w}^T\mathbf{x}_i + b) \ge 1 - \xi_i, \qquad i = 1,\dots,n.
\tag{6.1}
$$

Геометрическая интерпретация $\xi_i$ зависит от окончательного решения задачи (см. раздел 7), поскольку в процессе оптимизации значения $\xi_i$ будут выбраны минимально возможными для выполнения ограничений. В оптимальной точке справедливо соотношение $\xi_i = \max(0,\, 1 - y_i(\mathbf{w}^T\mathbf{x}_i + b))$, поэтому можно выделить три характерных режима:

- **$\xi_i = 0$** — точка лежит на правильной стороне и либо находится на границе зазора, либо дальше неё. Ограничение выполнено в исходном виде $y_i(\mathbf{w}^T\mathbf{x}_i + b) \ge 1$.
- **$0 < \xi_i \le 1$** — точка попадает внутрь разделяющей полосы, но всё ещё классифицируется верно, так как $y_i(\mathbf{w}^T\mathbf{x}_i + b) = 1 - \xi_i \ge 0$.
- **$\xi_i > 1$** — точка классифицируется ошибочно, поскольку $y_i(\mathbf{w}^T\mathbf{x}_i + b) = 1 - \xi_i < 0$.

Таким образом, slack‑переменная измеряет «степень нарушения» границы зазора в единицах функционального зазора. Сумма $\sum \xi_i$ выступает как оценка суммарных потерь, связанных с отступлением от идеальной разделимости.


## 7. Прямая задача soft‑margin SVM

Естественно объединить две цели — максимизацию зазора (т.е. минимизацию $\|\mathbf{w}\|$) и минимизацию суммарного штрафа за нарушения — в одной целевой функции. Введём **параметр компромисса** $C > 0$ и сформулируем прямую задачу **soft‑margin SVM**:
$$
\boxed{\begin{aligned}
\min_{\mathbf{w},\,b,\,\boldsymbol{\xi}} \quad & \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \xi_i \\
\text{при ограничениях} \quad & y_i(\mathbf{w}^T\mathbf{x}_i + b) \ge 1 - \xi_i, \quad i = 1,\dots,n, \\
& \xi_i \ge 0, \quad i = 1,\dots,n.
\end{aligned}}
\tag{7.1}
$$

Параметр $C$ управляет компромиссом между двумя конкурирующими требованиями. Рассмотрим крайние случаи:

- **$C \to 0$** делает штраф за нарушения пренебрежимо малым. Оптимизация сосредотачивается на минимизации $\|\mathbf{w}\|^2$, что даёт максимально широкий зазор ценой большого числа нарушений — возникает опасность недообучения.
- **$C \to \infty$** делает недопустимыми даже малые нарушения, и задача всё более приближается к hard‑margin SVM. Если данные линейно разделимы, решение будет определяться единичными выбросами, что ведёт к переобучению. Если же данные неразделимы, то при $C \to \infty$ допустимого решения не существует вовсе — ограничения становятся несовместными. Поэтому на практике используют конечные значения $C$, подбирая их, например, по кросс-валидации.

Задача (7.1) остаётся задачей выпуклого квадратичного программирования с линейными ограничениями. Гладкая квадратичная часть $\frac{1}{2}\|\mathbf{w}\|^2$ гарантирует единственность решения по $\mathbf{w}$, а линейный штраф $\sum \xi_i$ (в отличие от квадратичного $\sum \xi_i^2$) приводит к разреженности по нарушениям: в оптимуме многие $\xi_i$ окажутся равными нулю, поскольку для точек, удовлетворяющих жёсткому ограничению, любое ненулевое $\xi_i$ только увеличило бы целевую функцию без необходимости.

> **Углублённый взгляд: связь с hinge loss и структурным риском**  
> Задачу (7.1) можно переписать в безусловной форме. При фиксированных $\mathbf{w}, b$ оптимальное значение каждой переменной $\xi_i$ есть $\max(0,\, 1 - y_i(\mathbf{w}^T\mathbf{x}_i + b))$, поскольку любое превышение этого порога только увеличило бы целевую функцию. Подставляя эти выражения, получаем эквивалентную задачу
> $$
> \min_{\mathbf{w},b} \left\{ \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \max\bigl(0,\, 1 - y_i(\mathbf{w}^T\mathbf{x}_i + b)\bigr) \right\}.
> $$
> Разделив целевую функцию на $C$ и обозначив $\lambda = \frac{1}{2C}$, приходим к форме «регуляризованный эмпирический риск»:
> $$
> \min_{\mathbf{w},b} \left\{ \sum_{i=1}^{n} \max\bigl(0,\, 1 - y_i f(\mathbf{x}_i)\bigr) + \lambda \|\mathbf{w}\|^2 \right\},
> $$
> где $f(\mathbf{x}) = \mathbf{w}^T\mathbf{x} + b$. Функция потерь $L(y, f(\mathbf{x})) = \max(0, 1 - y f(\mathbf{x}))$ называется **hinge loss** (шарнирная потеря). Она равна нулю, когда функциональный зазор не меньше $1$, и растёт линейно при его уменьшении, штрафуя как ошибочные, так и «неуверенные» правильные предсказания.  
> Эта запись раскрывает статистическую природу SVM. Слагаемое $\sum \max(0, 1 - y f)$ оценивает эмпирический риск, а $\lambda \|\mathbf{w}\|^2$ служит штрафом за сложность модели. В теории Вапника–Червоненкиса максимизация зазора эквивалентна минимизации VC‑размерности класса линейных классификаторов, а hinge loss является выпуклой мажорантой индикатора ошибки. Следовательно, soft‑margin SVM напрямую реализует **принцип структурной минимизации риска** (Structural Risk Minimization, SRM): выбирается модель, минимизирующая верхнюю границу ожидаемого риска, которая зависит как от эмпирических ошибок, так и от сложности класса функций.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from ipywidgets import interact, FloatLogSlider

# ================================================
# 1. Генерация данных с выбросом
# ================================================
np.random.seed(42)
X = np.array([[1,2],[2,3],[3,3],[2,1],[3,2],
              [5,5],[6,6],[7,5],[5,6],[6,5],
              [4, 4]])   # <-- выброс, класс +1, но лежит в "минусовой" области
y = np.array([-1,-1,-1,-1,-1, 1,1,1,1,1, 1])

# ================================================
# 2. Функция обучения и отрисовки
# ================================================
def plot_svm(C_value):
    clf = svm.SVC(kernel='linear', C=C_value)
    clf.fit(X, y)

    w = clf.coef_[0]
    b = clf.intercept_[0]
    margin = 1 / np.linalg.norm(w)

    # Коэффициенты alpha_i (только для опорных векторов)
    alpha_sv = clf.dual_coef_[0]   # y_i * alpha_i
    sv_idx = clf.support_
    alphas = np.abs(alpha_sv)       # сами alpha_i

    plt.figure(figsize=(8,6))
    # Все точки
    plt.scatter(X[:,0], X[:,1], c=y, cmap='bwr', s=80, edgecolors='k',
                label='Обучающие примеры')

    # Опорные векторы: свободные (0<alpha<C) — зелёная обводка,
    # связанные (alpha=C) — красная обводка
    free_mask = (alphas > 0) & (alphas < C_value - 1e-5)
    bounded_mask = (alphas >= C_value - 1e-5)

    plt.scatter(X[sv_idx[free_mask],0], X[sv_idx[free_mask],1],
                s=200, facecolors='none', edgecolors='green', linewidths=2,
                label='Свободные опорные векторы (0<α<C)')
    plt.scatter(X[sv_idx[bounded_mask],0], X[sv_idx[bounded_mask],1],
                s=200, facecolors='none', edgecolors='red', linewidths=2,
                label='Связанные опорные векторы (α=C)')

    # Границы зазора и гиперплоскость
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xx = np.linspace(xlim[0], xlim[1], 100)
    yy = np.linspace(ylim[0], ylim[1], 100)
    XX, YY = np.meshgrid(xx, yy)
    Z = clf.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contour(XX, YY, Z, colors='k', levels=[-1,0,1], linestyles=['--','-','--'])
    ax.set_title(f'Soft‑margin SVM при C = {C_value:.3f}\n'
                 f'Зазор = {margin:.3f} | Свободных ОВ: {np.sum(free_mask)}, '
                 f'Связанных: {np.sum(bounded_mask)}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# ================================================
# 3. Интерактивный слайдер
# ================================================
interact(plot_svm,
         C_value=FloatLogSlider(value=1.0, base=10, min=-2, max=2, step=0.1,
                                description='Параметр C'));


## 8. Двойственная задача soft‑margin SVM

Для построения двойственной задачи введём множители Лагранжа для каждого типа ограничений: $\alpha_i \ge 0$ для основных ограничений $y_i(\mathbf{w}^T\mathbf{x}_i + b) \ge 1 - \xi_i$ и $\mu_i \ge 0$ для условий неотрицательности $\xi_i \ge 0$. Лагранжиан принимает вид
$$
L(\mathbf{w}, b, \boldsymbol{\xi}, \boldsymbol{\alpha}, \boldsymbol{\mu}) = \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^{n} \xi_i - \sum_{i=1}^{n} \alpha_i \bigl( y_i(\mathbf{w}^T\mathbf{x}_i + b) - 1 + \xi_i \bigr) - \sum_{i=1}^{n} \mu_i \xi_i.
\tag{8.1}
$$
Двойственная функция есть $g(\boldsymbol{\alpha}, \boldsymbol{\mu}) = \inf_{\mathbf{w}, b, \boldsymbol{\xi}} L$. Условия минимума по прямым переменным получаются приравниванием нулю частных производных:
$$
\frac{\partial L}{\partial \mathbf{w}} = \mathbf{w} - \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i = \mathbf{0}
\quad\Longrightarrow\quad
\mathbf{w} = \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i,
\tag{8.2}
$$
$$
\frac{\partial L}{\partial b} = -\sum_{i=1}^{n} \alpha_i y_i = 0
\quad\Longrightarrow\quad
\sum_{i=1}^{n} \alpha_i y_i = 0,
\tag{8.3}
$$
$$
\frac{\partial L}{\partial \xi_i} = C - \alpha_i - \mu_i = 0
\quad\Longrightarrow\quad
\alpha_i + \mu_i = C, \quad i = 1,\dots,n.
\tag{8.4}
$$

Из (8.4) и условия $\mu_i \ge 0$ немедленно следует **верхняя граница** на $\alpha_i$: $0 \le \alpha_i \le C$. Подставим теперь (8.2)–(8.4) в лагранжиан. Слагаемые, содержащие $\xi_i$, дают
$$
\sum_i \xi_i (C - \alpha_i - \mu_i) = 0,
$$
и, следовательно, исчезают. Оставшаяся часть совпадает с выражением, полученным для hard‑margin SVM:
$$
L = \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j) - \sum_{i,j} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j) + \sum_{i} \alpha_i = \sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i=1}^{n}\sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j).
$$
Таким образом, двойственная задача soft‑margin SVM отличается от hard‑margin варианта только наличием ограничения $\alpha_i \le C$:
$$
\boxed{\begin{aligned}
\max_{\boldsymbol{\alpha}} \quad & \sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i=1}^{n}\sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j (\mathbf{x}_i^T \mathbf{x}_j) \\
\text{при ограничениях} \quad & 0 \le \alpha_i \le C, \quad i = 1,\dots,n, \\
& \sum_{i=1}^{n} \alpha_i y_i = 0.
\end{aligned}}
\tag{8.5}
$$

Структура оптимального решения становится богаче. Величина $\alpha_i$ теперь не только показывает, является ли точка опорным вектором, но и характеризует её положение относительно зазора:

- **$\alpha_i = 0$** — точка правильно классифицирована и лежит вне полосы зазора (или на её внешней стороне). Она не является опорным вектором и не участвует в формировании решения.
- **$0 < \alpha_i < C$** — точка находится **точно на границе зазора** (margin). Такие объекты называют **свободными опорными векторами** (free support vectors). Для них slack‑переменная $\xi_i = 0$ и выполняется строгое равенство $y_i(\mathbf{w}^T\mathbf{x}_i + b) = 1$.
- **$\alpha_i = C$** — точка лежит **внутри зазора** или ошибочно классифицирована (если $\xi_i > 1$). Её slack‑переменная, как правило, положительна. Эти объекты называют **связанными опорными векторами** (bounded support vectors). Верхняя граница $C$ ограничивает влияние таких «проблемных» точек на итоговое решение, не давая единичному выбросу полностью определить гиперплоскость.

## 9. Условия KKT для soft‑margin SVM

Оптимальное решение задачи (7.1) должно удовлетворять полной системе условий Каруша–Куна–Таккера, которая для данной задачи выглядит следующим образом.

1. **Стационарность лагранжиана:**
   $$
   \mathbf{w}^* = \sum_{i=1}^{n} \alpha_i^* y_i \mathbf{x}_i, \qquad
   \sum_{i=1}^{n} \alpha_i^* y_i = 0, \qquad
   \alpha_i^* + \mu_i^* = C, \quad \forall i.
   $$
2. **Прямая допустимость:**
   $$
   y_i\bigl((\mathbf{w}^*)^T\mathbf{x}_i + b^*\bigr) \ge 1 - \xi_i^*, \quad \xi_i^* \ge 0, \quad \forall i.
   $$
3. **Двойственная допустимость:**
   $$
   \alpha_i^* \ge 0, \quad \mu_i^* \ge 0, \quad \forall i \quad\Longrightarrow\quad 0 \le \alpha_i^* \le C.
   $$
4. **Условия дополняющей нежёсткости:**
   $$
   \alpha_i^* \Bigl( y_i\bigl((\mathbf{w}^*)^T\mathbf{x}_i + b^*\bigr) - 1 + \xi_i^* \Bigr) = 0, \quad \forall i,
   \tag{9.1}
   $$
   $$
   \mu_i^* \, \xi_i^* = 0, \quad \forall i.
   \tag{9.2}
   $$

Используя эти условия, можно строго обосновать классификацию точек по значению $\alpha_i$, приведённую выше.

- Если **$\alpha_i^* = 0$**, то из $\alpha_i^* + \mu_i^* = C$ следует $\mu_i^* = C > 0$, и условие (9.2) даёт $\xi_i^* = 0$. Тогда из (9.1) ограничение выполняется автоматически. Прямая допустимость гарантирует $y_i((\mathbf{w}^*)^T\mathbf{x}_i + b^*) \ge 1$ — объект правильно классифицирован и лежит вне зазора.
- Если **$0 < \alpha_i^* < C$**, то $\mu_i^* = C - \alpha_i^* > 0$, и вновь $\xi_i^* = 0$ из (9.2). Теперь (9.1) превращается в $y_i((\mathbf{w}^*)^T\mathbf{x}_i + b^*) - 1 = 0$, то есть точка лежит точно на границе зазора.
- Если **$\alpha_i^* = C$**, то $\mu_i^* = 0$, и условие (9.2) не накладывает ограничений на $\xi_i^*$ (кроме неотрицательности). Из (9.1) следует, что если $\xi_i^* > 0$, то скобка равна нулю, то есть $y_i((\mathbf{w}^*)^T\mathbf{x}_i + b^*) = 1 - \xi_i^* < 1$. Точка находится внутри полосы зазора, и при $\xi_i^* > 1$ она классифицируется ошибочно.

> **Углублённый взгляд: устойчивое вычисление смещения $b$ в soft‑margin**  
> Для любого свободного опорного вектора (индекс $s$ с $0 < \alpha_s < C$) имеем $\xi_s = 0$ и равенство $y_s(\mathbf{w}^T\mathbf{x}_s + b) = 1$. Отсюда смещение можно выразить как
> $$
> b = y_s - \sum_{j=1}^{n} \alpha_j y_j (\mathbf{x}_j^T \mathbf{x}_s).
> $$
> Однако единичный опорный вектор может давать неустойчивую оценку из‑за численных погрешностей или особенностей данных. На практике надёжнее усреднять $b$ по всем свободным опорным векторам:
> $$
> b^* = \frac{1}{|\mathcal{F}|} \sum_{s \in \mathcal{F}} \left( y_s - \sum_{j=1}^{n} \alpha_j y_j (\mathbf{x}_j^T \mathbf{x}_s) \right),
> \tag{9.3}
> $$
> где $\mathcal{F} = \{ i \mid 0 < \alpha_i < C \}$. Если множество $\mathcal{F}$ пусто (все опорные векторы достигли границы $C$), это сигнал о том, что модель чрезмерно «зажата» и значение $C$, вероятно, слишком мало. В таких случаях прибегают к более сложным эвристикам, например, к решению задачи минимизации невязок по $b$ с привлечением всех опорных векторов, что, однако, выходит за рамки канонического изложения.

Итоговое решающее правило для soft‑margin SVM сохраняет линейную форму
$$
f(\mathbf{x}) = \sum_{i=1}^{n} \alpha_i^* y_i (\mathbf{x}_i^T \mathbf{x}) + b^*, \qquad \hat{y} = \operatorname{sign}\!\bigl(f(\mathbf{x})\bigr),
$$
где суммирование фактически ведётся только по опорным векторам (как свободным, так и связанным). Принципиальное отличие от hard‑margin случая заключается в появлении верхней границы $C$ для множителей $\alpha_i$, что не позволяет выбросам доминировать в решении и делает модель работоспособной на неразделимых данных.

Таким образом, soft‑margin SVM элегантно расширяет геометрическую идею максимизации зазора на реальные, зашумлённые выборки, сохраняя выпуклость, разреженность опорных векторов и возможность ядерного обобщения, которому будет посвящён следующий раздел.



In [ ]:
# @title Проверка вычисления b через свободные опорные векторы
# Берём последний обученный классификатор (при C=1 или любом другом)
alpha_sv = clf.dual_coef_[0]
sv = clf.support_vectors_
y_sv = y[clf.support_]

# Свободные опорные векторы: 0 < alpha < C (с запасом)
tol = 1e-5
free = (np.abs(alpha_sv) > tol) & (np.abs(alpha_sv) < clf.C - tol)

if np.any(free):
    b_values = y_sv[free] - np.dot(sv[free], clf.coef_[0])
    b_manual = np.mean(b_values)
    print(f'b (sklearn): {clf.intercept_[0]:.6f}')
    print(f'b (усредн. по свободным ОВ): {b_manual:.6f}')
    print('Совпадают?', np.isclose(clf.intercept_[0], b_manual, atol=1e-4))
else:
    print('Нет свободных опорных векторов — C слишком мал или велик.')



## Вопросы для самопроверки

### Для начинающих

1. Для чего вводятся slack‑переменные $\xi_i$ в SVM и какие три режима их значений можно выделить?
2. Что означает ситуация $\xi_i > 1$? Как при этом классифицируется объект?
3. Какую роль играет параметр $C$ в задаче soft‑margin SVM? К чему приводит его неограниченное увеличение и уменьшение?
4. Чем ограничения двойственной задачи soft‑margin SVM отличаются от hard‑margin варианта?
5. Где находятся точки, для которых оптимальное значение $\alpha_i = C$?

### Для профессионалов

1. Выведите двойственную задачу soft‑margin SVM, стартуя с лагранжиана и учитывая оба набора ограничений. Покажите, как появляется верхняя граница $\alpha_i \le C$.
2. Докажите, используя условия стационарности по $\xi_i$, что $0 \le \alpha_i \le C$. Почему это ограничение критически важно при работе с неразделимыми данными?
3. Покажите эквивалентность прямой задачи soft‑margin SVM и задачи минимизации регуляризованного эмпирического риска с hinge loss. Как связаны параметры $C$ и $\lambda$?
4. Почему для вычисления смещения $b$ в soft‑margin SVM предпочтительно использовать только свободные опорные векторы ($0 < \alpha_i < C$), а не связанные ($\alpha_i = C$)? Каковы последствия отсутствия свободных опорных векторов?
5. В чём заключается принцип структурной минимизации риска (SRM) и как soft‑margin SVM его реализует? Какую роль здесь играет максимизация зазора?

## 10. Мотивация: от линейного к нелинейному разделению

Линейный SVM, рассмотренный в предыдущих разделах, строит разделяющую гиперплоскость. Однако реальные данные часто не допускают линейного разделения без ошибок — и дело не только в шуме, но и в самой структуре классов. Классический пример: в двумерном пространстве один класс лежит внутри круга, другой — снаружи. Никакая прямая не может разделить их без грубых ошибок.

Идея состоит в том, чтобы отобразить исходные векторы признаков в новое, **спрямляющее пространство** (feature space) $\mathcal{H}$ с помощью нелинейного преобразования
$$
\phi: \mathbb{R}^D \to \mathcal{H},
$$
а затем применить линейный SVM уже в $\mathcal{H}$. Если отображение $\phi$ выбрано удачно, данные в $\mathcal{H}$ окажутся линейно разделимыми. Геометрически это означает, что мы ищем гиперплоскость не в исходном пространстве, а в пространстве, где классы «распрямляются». Например, для кругового примера достаточно добавить квадрат расстояния от начала координат: $\phi(x_1, x_2) = (x_1, x_2, x_1^2 + x_2^2)$ — тогда точки внутреннего круга окажутся ниже некоторой плоскости, а внешнего — выше.

Запустите ячейку ниже, чтобы увидеть эту идею в действии: слева — линейный SVM в исходном пространстве (он не может разделить круги), справа — после добавления третьего признака данные становятся линейно разделимыми.

In [ ]:
# @title Визуализация нелинейного разделения: от 2D к 3D
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from mpl_toolkits.mplot3d import Axes3D

# 1. Генерация данных: класс -1 внутри круга радиуса 1, класс +1 в кольце между радиусами 2 и 3
np.random.seed(0)
n = 100
r_inner = np.random.rand(n) * 1.0
theta_inner = np.random.rand(n) * 2*np.pi
X_inner = np.column_stack([r_inner*np.cos(theta_inner), r_inner*np.sin(theta_inner)])
y_inner = -np.ones(n)

r_outer = 2 + np.random.rand(n) * 1.0
theta_outer = np.random.rand(n) * 2*np.pi
X_outer = np.column_stack([r_outer*np.cos(theta_outer), r_outer*np.sin(theta_outer)])
y_outer = np.ones(n)

X = np.vstack([X_inner, X_outer])
y = np.hstack([y_inner, y_outer])

# 2. Линейный SVM в исходном 2D пространстве
clf_2d = svm.SVC(kernel='linear', C=1)
clf_2d.fit(X, y)
w = clf_2d.coef_[0]
b = clf_2d.intercept_[0]

# 3. Отображение phi: (x1, x2) -> (x1, x2, x1^2 + x2^2)
X3 = np.column_stack([X[:,0], X[:,1], X[:,0]**2 + X[:,1]**2])
clf_3d = svm.SVC(kernel='linear', C=1)
clf_3d.fit(X3, y)

# 4. Построение графиков
fig = plt.figure(figsize=(12,5))

# Слева — исходные данные и линейная граница
ax1 = fig.add_subplot(1,2,1)
ax1.scatter(X[y==-1,0], X[y==-1,1], c='blue', s=20, label='Класс -1 (внутри)')
ax1.scatter(X[y==1,0], X[y==1,1], c='red', s=20, label='Класс +1 (снаружи)')

# Граница: w1*x1 + w2*x2 + b = 0
x1_line = np.linspace(-3, 3, 10)
x2_line = (-w[0]*x1_line - b) / w[1]
ax1.plot(x1_line, x2_line, 'k-', lw=2, label='Граница SVM')
ax1.set_xlim(-3, 3)
ax1.set_ylim(-3, 3)
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.set_title('Линейный SVM в исходных признаках\n(неразделимы)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Справа — 3D данные и разделяющая гиперплоскость
ax2 = fig.add_subplot(1,2,2, projection='3d')
ax2.scatter(X3[y==-1,0], X3[y==-1,1], X3[y==-1,2], c='blue', s=20, label='Класс -1')
ax2.scatter(X3[y==1,0], X3[y==1,1], X3[y==1,2], c='red', s=20, label='Класс +1')

# Построим плоскость: w1*x1 + w2*x2 + w3*x3 + b = 0
w3 = clf_3d.coef_[0]
b3 = clf_3d.intercept_[0]
xx, yy = np.meshgrid(np.linspace(-3,3,10), np.linspace(-3,3,10))
zz = (-w3[0]*xx - w3[1]*yy - b3) / w3[2]
ax2.plot_surface(xx, yy, zz, alpha=0.3, color='gray')
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_zlabel('$x_1^2 + x_2^2$')
ax2.set_title('После добавления $x_1^2+x_2^2$\nданные линейно разделимы')
ax2.legend()
plt.tight_layout()
plt.show()


## 11. Ядровой трюк

Прямое построение $\phi$ и работа в $\mathcal{H}$ могут быть вычислительно нереализуемы, если $\mathcal{H}$ имеет высокую или бесконечную размерность. Однако и в двойственной задаче soft‑margin SVM (8.5), и в решающем правиле (5.2) векторы $\mathbf{x}_i$ входят только через скалярные произведения $\mathbf{x}_i^T \mathbf{x}_j$. При переходе к $\mathcal{H}$ эти произведения заменяются на $\phi(\mathbf{x}_i)^T \phi(\mathbf{x}_j)$. **Ядровой трюк** (kernel trick) состоит в том, чтобы вычислять скалярное произведение в $\mathcal{H}$ напрямую с помощью **функции ядра**
$$
K(\mathbf{x}, \mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{H}},
$$
не строя явно ни отображение $\phi$, ни координаты векторов в $\mathcal{H}$.

Подставив ядро в двойственную задачу, получаем:
$$
\boxed{\begin{aligned}
\max_{\boldsymbol{\alpha}} \quad & \sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i=1}^{n}\sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j K(\mathbf{x}_i, \mathbf{x}_j) \\
\text{при ограничениях} \quad & 0 \le \alpha_i \le C, \quad i = 1,\dots,n, \\
& \sum_{i=1}^{n} \alpha_i y_i = 0.
\end{aligned}}
\tag{11.1}
$$
Решающее правило для нового объекта $\mathbf{x}$ принимает вид
$$
f(\mathbf{x}) = \sum_{i=1}^{n} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b, \qquad \hat{y} = \operatorname{sign}\!\bigl(f(\mathbf{x})\bigr).
\tag{11.2}
$$
Вычисление $b$ производится аналогично линейному случаю — через любой свободный опорный вектор (тот, для которого $0 < \alpha_i < C$), с заменой скалярного произведения на ядро:
$$
b = y_s - \sum_{j=1}^{n} \alpha_j y_j K(\mathbf{x}_j, \mathbf{x}_s).
\tag{11.3}
$$
На практике $b$ усредняют по всем свободным опорным векторам.

Таким образом, всё обучение и классификация сводятся к вычислению значений ядра между парами объектов. Это позволяет применять SVM с нелинейными границами, оставаясь в рамках выпуклой оптимизации той же размерности $n$, что и в линейном случае.



In [ ]:
# @title Проверка ядрового трюка: явное отображение vs. ядро
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVC

# Две произвольные точки
x1 = np.array([1.0, 2.0])
x2 = np.array([3.0, -1.0])

# Полиномиальное ядро (gamma=1, r=0, d=2)
def poly_kernel(a, b):
    return (a @ b) ** 2

# Явное отображение phi: (x1, x2) -> (x1^2, sqrt(2)*x1*x2, x2^2)
phi = lambda x: np.array([x[0]**2, np.sqrt(2)*x[0]*x[1], x[1]**2])

print("Скалярное произведение через phi:", phi(x1) @ phi(x2))
print("Значение полиномиального ядра:       ", poly_kernel(x1, x2))
print("Совпадают:", np.isclose(phi(x1) @ phi(x2), poly_kernel(x1, x2)))

# Теперь на примере SVM: сравним линейный SVM на phi-признаках и SVM с полиномиальным ядром
np.random.seed(0)
X = np.random.randn(20, 2)
y = np.sign(X[:, 0]**2 + X[:, 1]**2 - 1.5)

# Линейный SVM на явных phi-признаках
X_phi = np.column_stack([X[:, 0]**2, np.sqrt(2)*X[:, 0]*X[:, 1], X[:, 1]**2])
svm_phi = SVC(kernel='linear', C=1)
svm_phi.fit(X_phi, y)

# SVM с полиномиальным ядром
svm_kernel = SVC(kernel='poly', degree=2, gamma=1, coef0=0, C=1)
svm_kernel.fit(X, y)

# Сравним предсказания на всех точках
print("\nПредсказания совпадают?",
      np.all(svm_phi.predict(X_phi) == svm_kernel.predict(X)))

## 12. Теорема Мерсера и свойства ядер

Чтобы некоторая функция $K$ могла быть использована в качестве ядра, она должна соответствовать скалярному произведению в некотором гильбертовом пространстве. Критерий этого даётся условием положительной полуопределённости.

**Определение.** Функция $K: \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ называется **положительно полуопределённым ядром** (positive semidefinite kernel), если для любого конечного набора точек $\mathbf{x}_1, \dots, \mathbf{x}_m \in \mathcal{X}$ матрица $\mathbf{K}$ с элементами $K_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$ является симметричной и положительно полуопределённой, то есть
$$
\sum_{i=1}^{m} \sum_{j=1}^{m} c_i c_j K(\mathbf{x}_i, \mathbf{x}_j) \ge 0
$$
для любых вещественных $c_1, \dots, c_m$.

**Теорема Мерсера** (упрощённая формулировка). Пусть $\mathcal{X}$ — компактное подмножество $\mathbb{R}^D$, а $K: \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ — симметричная непрерывная функция, являющаяся положительно полуопределённым ядром. Тогда существует гильбертово пространство $\mathcal{H}$ и отображение $\phi: \mathcal{X} \to \mathcal{H}$ такие, что для всех $\mathbf{x}, \mathbf{x}' \in \mathcal{X}$
$$
K(\mathbf{x}, \mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{H}}.
$$

На практике теорема Мерсера гарантирует, что любое положительно полуопределённое ядро неявно работает со скалярным произведением в некотором спрямляющем пространстве $\mathcal{H}$, даже если мы никогда не строим его в явном виде. Это делает корректным применение SVM с таким ядром: задача (11.1) остаётся выпуклой, а решение соответствует максимизации зазора в $\mathcal{H}$.

In [ ]:
# @title Проверка положительной полуопределённости RBF-ядра
from sklearn.metrics.pairwise import rbf_kernel
import numpy as np

X = np.random.randn(10, 2)
K = rbf_kernel(X, gamma=1.0)

# Симметричность
print("Симметрична:", np.allclose(K, K.T))

# Все собственные значения >= 0 (с учётом погрешности)
eigvals = np.linalg.eigvalsh(K)
print("Минимальное собственное значение:", eigvals.min())
print("Все собств. значения >= 0:", np.all(eigvals > -1e-10))


## 13. Стандартные ядра и их геометрический смысл

Рассмотрим наиболее распространённые семейства ядер.

**Линейное ядро**  
$K(\mathbf{x}, \mathbf{x}') = \mathbf{x}^T \mathbf{x}'$. Это вырожденный случай ядрового SVM, в точности соответствующий линейному классификатору в исходном пространстве.

**Полиномиальное ядро**  
$$
K(\mathbf{x}, \mathbf{x}') = (\gamma \mathbf{x}^T \mathbf{x}' + r)^d, \quad d \in \mathbb{N}, \;\gamma > 0, \; r \ge 0.
$$
Спрямляющее пространство $\mathcal{H}$ состоит из всех мономов от исходных признаков степени не выше $d$. Параметр $\gamma$ масштабирует скалярное произведение, а $r$ добавляет возможность регулировать вклад членов низких степеней. При $d=2$ и $r=0$ ядро порождает все квадратичные признаки и их парные взаимодействия.

**Радиально-базисное ядро (RBF)**  
$$
K(\mathbf{x}, \mathbf{x}') = \exp\!\bigl(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2\bigr), \quad \gamma > 0.
$$
Это одно из самых популярных нелинейных ядер. Оно соответствует бесконечномерному гильбертову пространству (в чём можно убедиться, разложив экспоненту в ряд Тейлора). Геометрически значение ядра близко к $1$ для близких точек и экспоненциально спадает до нуля при увеличении расстояния между ними. Параметр $\gamma$ определяет «радиус влияния» каждой точки: чем больше $\gamma$, тем более локальной становится классификация.

**Сигмоидальное ядро**  
$$
K(\mathbf{x}, \mathbf{x}') = \tanh(\gamma \mathbf{x}^T \mathbf{x}' + r).
$$
Оно напоминает функцию активации нейронных сетей, но не при всех параметрах удовлетворяет условию положительной полуопределённости. Тем не менее, на практике его иногда используют, особенно когда данные имеют определённую «архитектурную» структуру.

> **Углублённый взгляд: явный вид спрямляющего отображения для полиномиального ядра**  
> Рассмотрим полиномиальное ядро с $d=2$, $\gamma=1$, $r=0$: $K(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^T \mathbf{x}')^2$. Пусть $\mathbf{x} = (x_1, x_2)^T$. Определим отображение
> $$
> \phi(\mathbf{x}) = (x_1^2,\; \sqrt{2}x_1 x_2,\; x_2^2)^T.
> $$
> Тогда скалярное произведение
> $$
> \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle = x_1^2 (x_1')^2 + 2 x_1 x_2 x_1' x_2' + x_2^2 (x_2')^2 = (x_1 x_1' + x_2 x_2')^2 = (\mathbf{x}^T \mathbf{x}')^2.
> $$
> Таким образом, $\phi$ действительно отображает двумерные векторы в трёхмерное пространство квадратичных признаков.  
> В общем случае для ядра $K(\mathbf{x}, \mathbf{x}') = (\gamma \mathbf{x}^T \mathbf{x}' + r)^d$ отображение $\phi$ перечисляет все мономы вида
> $$
> \sqrt{ \frac{d!}{k_0!\,k_1!\,\cdots\,k_D!}\,\gamma^{d-k_0}\,r^{k_0} }\; x_{i_1}\cdots x_{i_{d-k_0}},
> $$
> где $k_0$ — количество «константных» множителей $r$, а сумма остальных степеней $k_1+\dots+k_D = d-k_0$ (мультиномиальный коэффициент). Размерность спрямляющего пространства $\mathcal{H}$ равна числу мономов степени не выше $d$ от $D$ признаков, то есть $\binom{d + D}{D}$; при больших $d$ и $D$ она становится огромной, что и оправдывает применение ядрового трюка.

In [ ]:
# @title Влияние γ на границу SVM с RBF‑ядром
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from ipywidgets import interact, FloatLogSlider

# Генерация данных: две перемешанные "луны"
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=100, noise=0.2, random_state=42)

def plot_rbf(gamma):
    clf = svm.SVC(kernel='rbf', gamma=gamma, C=10)
    clf.fit(X, y)

    # Сетка для рисования границы
    xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200),
                         np.linspace(-1, 1.5, 200))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=(7,5))
    plt.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolors='k')
    plt.contour(xx, yy, Z, levels=[-1,0,1], linestyles=['--','-','--'],
                colors='k', alpha=0.7)
    plt.scatter(clf.support_vectors_[:,0], clf.support_vectors_[:,1],
                s=200, facecolors='none', edgecolors='k', linewidths=2)
    plt.title(f'RBF SVM, γ = {gamma:.3f}\n'
              f'Число опорных векторов: {len(clf.support_vectors_)}')
    plt.grid(True, alpha=0.3)
    plt.show()

interact(plot_rbf,
         gamma=FloatLogSlider(value=1.0, base=10, min=-2, max=2, step=0.1,
                              description='γ'));

## 14. Выбор ядра и его параметров

Выбор ядра — одна из ключевых методологических задач при использовании SVM. Общие рекомендации таковы:

- **Линейное ядро** стоит применять, когда данные априори приблизительно линейно разделимы, а размерность признаков велика (например, в классификации текстов). Оно самое быстрое и интерпретируемое.
- **RBF‑ядро** выступает как универсальный выбор «по умолчанию»: оно способно аппроксимировать любую гладкую границу, особенно при разумном подборе $\gamma$. При малом $\gamma$ граница получается гладкой и простой (ближе к линейной), при большом $\gamma$ — сильно изрезанной, подстраивающейся под отдельные точки. Чрезмерное увеличение $\gamma$ ведёт к переобучению: вокруг каждого опорного вектора образуется узкий «островок» его класса.
- **Полиномиальное ядро** полезно, когда есть основания предполагать полиномиальные зависимости между признаками. Однако большое $d$ может привести к численной нестабильности (очень большие значения ядра) и переобучению; обычно $d \le 3$.
- **Сигмоидальное ядро** применяют реже из‑за отсутствия гарантий выпуклости; оно может давать хорошие результаты лишь при специально подобранных $\gamma$ и $r$.

Параметры $C$ и $\gamma$ (или $d$ для полиномиального ядра) подбираются с помощью кросс‑валидации. Поскольку эти параметры меняются в широких пределах, перебор ведут по логарифмической сетке: например, $C \in \{10^{-3}, 10^{-2}, \dots, 10^{3}\}$, $\gamma \in \{10^{-4}, 10^{-3}, \dots, 10^{1}\}$. На практике для этого удобно использовать готовые инструменты, такие как `GridSearchCV` из библиотеки scikit‑learn, автоматически перебирающие заданные комбинации и оценивающие качество по нескольким фолдам. Важно помнить, что увеличение $C$ и $\gamma$ делает модель более сложной: $C$ разрешает меньше нарушений зазора, а $\gamma$ сужает зону влияния опорных векторов. Их совместная настройка ищет оптимальный баланс между смещением и разбросом, избегая как недообучения, так и переобучения. Более детально этот процесс будет показан в практической части главы.

Таким образом, ядерный SVM объединяет геометрическую интуицию максимизации зазора с гибкостью нелинейного моделирования, оставаясь в рамках строгой оптимизационной теории. Он стал одним из самых мощных и популярных методов машинного обучения до эпохи глубоких нейронных сетей и до сих пор сохраняет свои позиции в задачах с умеренным числом признаков и высокими требованиями к интерпретируемости.


Ячейка Markdown (опционально):
> Запустите ячейку ниже, чтобы увидеть, как перебор параметров по логарифмической сетке помогает выбрать лучшую модель. На графике показана точность кросс‑валидации для каждой пары \((C, \gamma)\).


In [ ]:
# @title GridSearchCV: подбор C и γ для RBF SVM
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Данные (те же "луны")
X, y = make_moons(n_samples=100, noise=0.2, random_state=42)

# Логарифмическая сетка
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 10, 100]
}

# Поиск с 5-кратной кросс-валидацией
svc = SVC(kernel='rbf')
grid = GridSearchCV(svc, param_grid, cv=5, scoring='accuracy')
grid.fit(X, y)

# Тепловая карта средних точностей
scores = grid.cv_results_['mean_test_score'].reshape(len(param_grid['C']), len(param_grid['gamma']))

plt.figure(figsize=(7,5))
plt.imshow(scores, interpolation='nearest', cmap='viridis')
plt.colorbar(label='Accuracy')
plt.xticks(np.arange(len(param_grid['gamma'])), param_grid['gamma'])
plt.yticks(np.arange(len(param_grid['C'])), param_grid['C'])
plt.xlabel('gamma')
plt.ylabel('C')
plt.title('Средняя точность кросс‑валидации')
# Аннотации
for i in range(len(param_grid['C'])):
    for j in range(len(param_grid['gamma'])):
        plt.text(j, i, f'{scores[i,j]:.3f}', ha='center', va='center', color='white' if scores[i,j] < 0.8 else 'black')
plt.show()

print("Лучшие параметры:", grid.best_params_)
print("Лучшая точность кросс‑валидации:", grid.best_score_)



## Вопросы для самопроверки

### Для начинающих

1. Что такое функция ядра в SVM и какую роль она выполняет?
2. Почему ядровой трюк позволяет строить нелинейные классификаторы, не увеличивая вычислительную сложность оптимизации?
3. Назовите три основных типа нелинейных ядер, используемых в SVM, и кратко опишите их.
4. Как параметр $\gamma$ в RBF‑ядре влияет на форму разделяющей границы? К чему приводит его чрезмерное увеличение?
5. Что утверждает теорема Мерсера? Какую гарантию она даёт при использовании ядра в SVM?

### Для профессионалов

1. Дайте формальное определение положительно полуопределённого ядра и объясните, как проверяется это свойство для конкретной функции $K$.
2. Докажите, что RBF‑ядро $K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$ соответствует бесконечномерному спрямляющему пространству. (Указание: разложите экспоненту в ряд Тейлора по скалярному произведению.)
3. Постройте в явном виде спрямляющее отображение $\phi$ для полиномиального ядра $(\mathbf{x}^T \mathbf{x}')^3$ в $\mathbb{R}^2$ и убедитесь, что $\langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle$ совпадает с исходным ядром.
4. При каких условиях сигмоидальное ядро $\tanh(\gamma \mathbf{x}^T \mathbf{x}' + r)$ удовлетворяет условиям теоремы Мерсера? Почему на практике это не всегда препятствие для его использования?
5. Обсудите компромисс между смещением и разбросом (bias‑variance trade‑off) при выборе параметра $\gamma$ в RBF‑ядре. Как этот выбор соотносится с регуляризационным параметром $C$?

## 15. Проблема масштабируемости двойственной задачи

Двойственная задача SVM, даже с ядром, остаётся выпуклой и имеет размерность, равную числу наблюдений $n$. Это означает, что при прямом применении стандартных решателей квадратичного программирования (QP) требуется хранить и обрабатывать ядерную матрицу $\mathbf{K} \in \mathbb{R}^{n \times n}$, что требует $O(n^2)$ памяти и $O(n^3)$ времени для точного решения. Для выборок объёмом в десятки и сотни тысяч примеров такой подход неприемлем.

Идея **декомпозиционных методов** состоит в том, чтобы на каждом шаге оптимизировать лишь небольшое подмножество переменных $\alpha_i$, фиксируя остальные. Самый радикальный вариант — алгоритм **Sequential Minimal Optimization (SMO)**, предложенный Платтом (1998), — работает ровно с двумя множителями за итерацию. Выбор пары продиктован тем, что ограничение $\sum_{k=1}^{n} \alpha_k y_k = 0$ при фиксации всех переменных, кроме двух, позволяет выразить одну из них через другую и решить получающуюся одномерную задачу аналитически. Это избавляет от необходимости вызывать внешний QP‑солвер и делает каждый шаг чрезвычайно быстрым.

## 16. Аналитическое решение для двух множителей

Пусть на текущей итерации выбраны индексы $i$ и $j$. Зафиксируем все множители $\alpha_k$ для $k \neq i,j$. Тогда ограничение $\sum_k \alpha_k y_k = 0$ превращается в
$$
\alpha_i y_i + \alpha_j y_j = -\sum_{k \neq i,j} \alpha_k y_k \equiv \Gamma,
$$
где $\Gamma$ — константа. Умножая обе части на $y_i$ и учитывая, что $y_i^2 = 1$, получаем
$$
\alpha_i + y_i y_j \alpha_j = \Gamma y_i \quad \Longrightarrow \quad \alpha_i = \Gamma y_i - y_i y_j \alpha_j.
\tag{16.1}
$$
Таким образом, $\alpha_i$ линейно выражается через $\alpha_j$.

Подставим это выражение в целевую функцию двойственной задачи
$$
\Psi(\boldsymbol{\alpha}) = \sum_{k=1}^{n} \alpha_k - \frac{1}{2}\sum_{k,l=1}^{n} \alpha_k \alpha_l y_k y_l K(\mathbf{x}_k, \mathbf{x}_l)
$$
и сгруппируем слагаемые, зависящие только от $\alpha_j$. После алгебраических преобразований (которые мы воспроизведём ниже) получится квадратичная функция от одной переменной:
$$
\Psi(\alpha_j) = \frac{1}{2} \eta \alpha_j^2 + \beta \alpha_j + \text{const},
$$
где $\eta = K_{ii} + K_{jj} - 2K_{ij}$, а $\beta$ выражается через старые значения ошибок. Максимум такой функции при $\eta > 0$ (что гарантировано положительной полуопределённостью ядра) достигается в точке
$$
\alpha_j^{\text{new, unc}} = \alpha_j^{\text{old}} + \frac{y_j(E_i - E_j)}{\eta}.
\tag{16.2}
$$
Здесь $E_k = f(\mathbf{x}_k) - y_k$ — ошибка модели на $k$-м объекте в текущей конфигурации, а $f(\mathbf{x})$ — решающая функция, вычисленная по всем $\alpha_k$.

> **Вывод формулы (16.2)**
> Развернём целевую функцию, изолируя члены с $\alpha_i$ и $\alpha_j$:
> $$
> \begin{aligned}
> \Psi &= \sum_{k} \alpha_k - \frac{1}{2}\sum_{k,l} \alpha_k \alpha_l y_k y_l K_{kl} \\
> &= \alpha_i + \alpha_j + \sum_{k \neq i,j} \alpha_k \\
> &\quad - \frac{1}{2} \Big( \alpha_i^2 K_{ii} + \alpha_j^2 K_{jj} + 2\alpha_i\alpha_j y_i y_j K_{ij} \Big) \\
> &\quad - \alpha_i y_i \sum_{k \neq i,j} \alpha_k y_k K_{ik} - \alpha_j y_j \sum_{k \neq i,j} \alpha_k y_k K_{jk} + \text{const}.
> \end{aligned}
> $$
> Используем (16.1), чтобы исключить $\alpha_i$, и замечаем, что
> $$
> f(\mathbf{x}_k) = \sum_{l} \alpha_l y_l K_{lk} + b,
> $$
> откуда ошибка $E_k = f(\mathbf{x}_k) - y_k$. После подстановки и упрощений получаем
> $$
> \Psi(\alpha_j) = -\frac{1}{2} (K_{ii} + K_{jj} - 2K_{ij}) \alpha_j^2 + \bigl[ (E_i - E_j) y_j + (K_{ii} + K_{jj} - 2K_{ij}) \alpha_j^{\text{old}} \bigr] \alpha_j + \text{const}.
> $$
> Приравнивая производную к нулю, приходим к (16.2).

Однако полученное аналитическое значение $\alpha_j^{\text{new, unc}}$ может выходить за допустимые пределы, диктуемые условиями $0 \le \alpha_i, \alpha_j \le C$ и линейным ограничением (16.1). Необходимо выполнить **клиппинг** (clipping) в интервал $[L, H]$, где границы зависят от меток $y_i, y_j$ и текущих значений $\alpha_i^{\text{old}}, \alpha_j^{\text{old}}$.

Из (16.1) и ограничений $0 \le \alpha_i \le C$, $0 \le \alpha_j \le C$ получаем:

- **Если $y_i \neq y_j$:** условие $\alpha_i = \Gamma y_i - y_i y_j \alpha_j = \Gamma y_i + \alpha_j$. Отсюда
  $$
  L = \max(0, \alpha_j^{\text{old}} - \alpha_i^{\text{old}}), \qquad
  H = \min(C, C + \alpha_j^{\text{old}} - \alpha_i^{\text{old}}).
  $$
- **Если $y_i = y_j$:** тогда $\alpha_i = \Gamma y_i - \alpha_j$. Отсюда
  $$
  L = \max(0, \alpha_i^{\text{old}} + \alpha_j^{\text{old}} - C), \qquad
  H = \min(C, \alpha_i^{\text{old}} + \alpha_j^{\text{old}}).
  $$

Таким образом,
$$
\alpha_j^{\text{new}} = \begin{cases}
H, & \text{если } \alpha_j^{\text{new, unc}} > H, \\
\alpha_j^{\text{new, unc}}, & \text{если } L \le \alpha_j^{\text{new, unc}} \le H, \\
L, & \text{если } \alpha_j^{\text{new, unc}} < L.
\end{cases}
\tag{16.3}
$$
После обновления $\alpha_j$ значение $\alpha_i$ корректируется согласно (16.1):
$$
\alpha_i^{\text{new}} = \alpha_i^{\text{old}} + y_i y_j (\alpha_j^{\text{old}} - \alpha_j^{\text{new}}).
\tag{16.4}
$$

## 17. Обновление смещения $b$

Смещение $b$ должно удовлетворять условиям KKT для опорных векторов. После шага по $\alpha_i, \alpha_j$ его пересчитывают так, чтобы для свободных опорных векторов выполнялось равенство $y_k f(\mathbf{x}_k) = 1$.

Если после обновления $0 < \alpha_i^{\text{new}} < C$, то из условия $y_i f(\mathbf{x}_i) = 1$ получаем
$$
b_i^{\text{new}} = -E_i - y_i K_{ii} (\alpha_i^{\text{new}} - \alpha_i^{\text{old}}) - y_j K_{ji} (\alpha_j^{\text{new}} - \alpha_j^{\text{old}}) + b^{\text{old}}.
\tag{17.1}
$$
Аналогично, если $0 < \alpha_j^{\text{new}} < C$, то
$$
b_j^{\text{new}} = -E_j - y_i K_{ij} (\alpha_i^{\text{new}} - \alpha_i^{\text{old}}) - y_j K_{jj} (\alpha_j^{\text{new}} - \alpha_j^{\text{old}}) + b^{\text{old}}.
\tag{17.2}
$$

Если оба множителя оказались свободными, $b$ полагают равным среднему $(b_i + b_j)/2$. Если ни один из них не свободен, $b$ оставляют прежним (в программной реализации можно также выбирать $b$ между $b_i$ и $b_j$).

## 18. Реализация класса `SimpleSVM` на Python

Приведём полную реализацию упрощённого алгоритма SMO, достаточную для обучения небольших выборок. Код снабжён комментариями и ориентирован на ясность, а не на рекордную производительность.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

class SimpleSVM:
    def __init__(self, C=1.0, kernel='rbf', gamma=1.0, tol=1e-3, max_iter=1000):
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.tol = tol
        self.max_iter = max_iter
        self.alpha = None
        self.support_ = None
        self.b = 0.0

    def _kernel(self, X1, X2):
        if self.kernel == 'linear':
            return X1 @ X2.T
        elif self.kernel == 'rbf':
            sq = np.sum(X1**2, axis=1).reshape(-1,1) + np.sum(X2**2, axis=1) - 2 * (X1 @ X2.T)
            return np.exp(-self.gamma * sq)
        else:
            raise ValueError("Unknown kernel")

    def _decision_function(self, X):
        # X: (m, d), self.X: (n, d)
        K = self._kernel(self.X, X)
        return (self.alpha * self.y) @ K + self.b

    def _take_step(self, i, j):
        if i == j:
            return False
        alpha_i_old, alpha_j_old = self.alpha[i], self.alpha[j]
        y_i, y_j = self.y[i], self.y[j]

        # Вычисляем ошибки E_i, E_j
        f_i = self._decision_function(self.X[[i]])[0]
        f_j = self._decision_function(self.X[[j]])[0]
        E_i = f_i - y_i
        E_j = f_j - y_j

        # Границы L, H
        if y_i != y_j:
            L = max(0, alpha_j_old - alpha_i_old)
            H = min(self.C, self.C + alpha_j_old - alpha_i_old)
        else:
            L = max(0, alpha_i_old + alpha_j_old - self.C)
            H = min(self.C, alpha_i_old + alpha_j_old)

        if L == H:
            return False

        # Второй член η
        K_ii = self._kernel(self.X[[i]], self.X[[i]])[0,0]
        K_jj = self._kernel(self.X[[j]], self.X[[j]])[0,0]
        K_ij = self._kernel(self.X[[i]], self.X[[j]])[0,0]
        eta = K_ii + K_jj - 2 * K_ij

        if eta <= 0:
            return False

        # Необрезанное обновление
        alpha_j_new_unc = alpha_j_old + y_j * (E_i - E_j) / eta

        # Клиппинг
        if alpha_j_new_unc > H:
            alpha_j_new = H
        elif alpha_j_new_unc < L:
            alpha_j_new = L
        else:
            alpha_j_new = alpha_j_new_unc

        if abs(alpha_j_new - alpha_j_old) < 1e-5:
            return False

        # Новое alpha_i
        alpha_i_new = alpha_i_old + y_i * y_j * (alpha_j_old - alpha_j_new)

        # Обновляем b
        b1 = -E_i - y_i * K_ii * (alpha_i_new - alpha_i_old) - y_j * K_ij * (alpha_j_new - alpha_j_old) + self.b
        b2 = -E_j - y_i * K_ij * (alpha_i_new - alpha_i_old) - y_j * K_jj * (alpha_j_new - alpha_j_old) + self.b

        if 0 < alpha_i_new < self.C:
            self.b = b1
        elif 0 < alpha_j_new < self.C:
            self.b = b2
        else:
            self.b = (b1 + b2) / 2.0

        self.alpha[i] = alpha_i_new
        self.alpha[j] = alpha_j_new

        # Обновляем кеш ошибок
        self.error_cache[i] = self._decision_function(self.X[[i]])[0] - y_i
        self.error_cache[j] = self._decision_function(self.X[[j]])[0] - y_j

        return True

    def fit(self, X, y):
        n = X.shape[0]
        self.X = X
        self.y = y.astype(float)
        self.alpha = np.zeros(n)
        self.b = 0.0
        self.error_cache = np.zeros(n)

        # Инициализация кеша ошибок
        for i in range(n):
            self.error_cache[i] = self._decision_function(X[[i]])[0] - y[i]

        num_changed = 0
        examine_all = True
        it = 0

        while it < self.max_iter and (num_changed > 0 or examine_all):
            num_changed = 0
            if examine_all:
                index_list = range(n)
            else:
                # индексы, где 0 < alpha < C
                index_list = np.where((self.alpha > 0) & (self.alpha < self.C))[0]

            for i in index_list:
                # Выбор j, максимизирующего |E_i - E_j|
                E_i = self.error_cache[i]
                if (self.y[i] * E_i < -self.tol and self.alpha[i] < self.C) or (self.y[i] * E_i > self.tol and self.alpha[i] > 0):
                    # Простейшая эвристика: случайный j, но для качества лучше максимизировать разницу ошибок
                    j = np.random.choice([idx for idx in range(n) if idx != i])
                    # Попытка улучшить: ищем максимальное |E_i - E_j|
                    max_diff = -1
                    for k in range(n):
                        if k == i: continue
                        diff = abs(E_i - self.error_cache[k])
                        if diff > max_diff:
                            max_diff = diff
                            j = k
                    if self._take_step(i, j):
                        num_changed += 1
            if examine_all:
                examine_all = False
            elif num_changed == 0:
                examine_all = True
            it += 1

        self.support_ = np.where(self.alpha > 1e-5)[0]
        self.support_vectors_ = X[self.support_]
        self.support_labels_ = y[self.support_]
        self.support_alpha_ = self.alpha[self.support_]
        return self

    def predict(self, X):
        return np.sign(self._decision_function(X))

    def decision_function(self, X):
        return self._decision_function(X)




## 19. Верификация реализации

### 19.1 Линейно разделимые данные и сравнение с sklearn

Сгенерируем простую двумерную линейно разделимую выборку и обучим на ней наш SimpleSVM с линейным ядром, а также `sklearn.svm.SVC` с `C = 1e10` (приближение hard‑margin). Сравним коэффициенты разделяющей прямой и опорные векторы.



In [ ]:
# @title 19.1 Линейно разделимые данные – визуализация
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs

# Генерация данных
np.random.seed(0)
X, y = make_blobs(n_samples=100, centers=2, random_state=42)
y = 2 * y - 1  # метки -1, 1

# Модели
model = SimpleSVM(C=1e10, kernel='linear', tol=1e-4, max_iter=2000)
model.fit(X, y)

clf = SVC(kernel='linear', C=1e10)
clf.fit(X, y)

# Построение сетки и контура
plt.figure(figsize=(8,6))
plt.scatter(X[:,0], X[:,1], c=y, cmap='bwr', alpha=0.7)
plt.scatter(model.support_vectors_[:,0], model.support_vectors_[:,1],
            s=100, facecolors='none', edgecolors='k', linewidths=1.5, label='Наши опорные')
plt.scatter(clf.support_vectors_[:,0], clf.support_vectors_[:,1],
            s=50, facecolors='green', edgecolors='k', label='sklearn опорные')

x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=['--','-','--'], colors='k')
plt.legend()
plt.title('Линейное ядро, hard‑margin')
plt.show()

# Сравнение коэффициентов
w_our = (model.alpha * model.y) @ model.X
w_sk = clf.coef_[0]
b_our = model.b
b_sk = clf.intercept_[0]
print(f"Наш w: {np.round(w_our, 4)}, b: {b_our:.4f}")
print(f"sklearn w: {np.round(w_sk, 4)}, b: {b_sk:.4f}")


Ожидаем, что значения будут близки (с точностью до численных погрешностей и разницы в алгоритме).

### 19.2 Сравнение на Breast Cancer (RBF‑ядро)

Загрузим датасет breast cancer, стандартизируем признаки, обучим обе модели с RBF‑ядром, $C=1$, $\gamma=1/30$ (или по умолчанию). Сравним точность (accuracy) и число опорных векторов.



In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
y = 2 * y - 1  # метки -1, 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Наш SVM
our_svm = SimpleSVM(C=1.0, kernel='rbf', gamma=1.0/X_train.shape[1], max_iter=2000, tol=1e-3)
start = time.time()
our_svm.fit(X_train, y_train)
our_time = time.time() - start
our_acc = np.mean(our_svm.predict(X_test) == y_test)
our_nSV = len(our_svm.support_)

# sklearn
sk_svm = SVC(C=1.0, kernel='rbf', gamma=1.0/X_train.shape[1])
start = time.time()
sk_svm.fit(X_train, y_train)
sk_time = time.time() - start
sk_acc = np.mean(sk_svm.predict(X_test) == y_test)
sk_nSV = len(sk_svm.support_)

print(f"Наша модель: точность {our_acc:.4f}, опорных векторов {our_nSV}, время {our_time:.2f}с")
print(f"Sklearn:     точность {sk_acc:.4f}, опорных векторов {sk_nSV}, время {sk_time:.2f}с")



### 19.3 Масштабирование: зависимость времени от размера выборки

Создадим синтетические выборки размером 200, 500, 1000 с двумя признаками, обучим обе модели (с RBF, фиксированные параметры) и построим график времени.




In [ ]:
sizes = [200, 500, 1000]
times_our = []
times_sk = []

for n in sizes:
    X_s, y_s = make_blobs(n_samples=n, centers=2, random_state=1, cluster_std=2.0)
    y_s = 2 * y_s - 1
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_s)

    start = time.time()
    svm = SimpleSVM(C=1.0, kernel='rbf', gamma=0.5, max_iter=500, tol=1e-3)
    svm.fit(X_s, y_s)
    times_our.append(time.time() - start)

    start = time.time()
    sk = SVC(C=1.0, kernel='rbf', gamma=0.5)
    sk.fit(X_s, y_s)
    times_sk.append(time.time() - start)

plt.figure(figsize=(8,5))
plt.plot(sizes, times_our, 'o-', label='SimpleSVM')
plt.plot(sizes, times_sk, 's-', label='sklearn SVC')
plt.xlabel('Число объектов')
plt.ylabel('Время обучения, с')
plt.title('Сравнение времени выполнения')
plt.legend()
plt.grid()
plt.show()



График наглядно покажет, что наша реализация медленнее библиотечной (которая использует оптимизированные алгоритмы и компиляцию), но демонстрирует качественно схожую кубическую зависимость. Это подчёркивает необходимость эффективных эвристик и методов, используемых в промышленных солверах.

---

## Вопросы для самопроверки

### Для начинающих

1. Почему в алгоритме SMO на каждом шаге оптимизируются ровно два множителя Лагранжа?
2. Что такое «кеширование ошибок» и зачем оно применяется?
3. Как метод `predict` в SVM использует опорные векторы? Зависят ли предсказания от объектов с $\alpha_i = 0$?
4. Какие точки обучающей выборки становятся опорными векторами? Какие значения $\alpha_i$ им соответствуют?

### Для профессионалов

1. Выведите в явном виде формулу для необрезанного обновления $\alpha_j^{\text{new, unc}}$, исходя из квадратичной аппроксимации двойственной функции по одной переменной.
2. Обоснуйте границы $L$ и $H$ при клиппинге для случаев $y_i \neq y_j$ и $y_i = y_j$; как они связаны с сохранением ограничения $\sum \alpha_k y_k = 0$ и $0 \le \alpha \le C$?
3. Как в SMO выбирается второй множитель $j$ для достижения максимального продвижения? Почему эвристика максимизации $|E_i - E_j|$ является разумной?
4. Объясните, как обновление смещения $b$ зависит от того, является ли опорный вектор свободным ($0 < \alpha < C$). Почему при отсутствии свободных векторов $b$ определяется неоднозначно?
5. Оцените сложность одного шага SMO и общую асимптотическую сложность алгоритма в зависимости от $n$ и числа итераций. Сравните с точным решением двойственной задачи.

## 20. Многоклассовая классификация SVM

Базовая формулировка SVM предназначена для бинарных задач. Однако на практике часто требуется различать $K \ge 3$ классов. Существуют две основные стратегии сведения многоклассовой задачи к набору бинарных — **One‑vs‑Rest (OvR)** и **One‑vs‑One (OvO)**, — а также более сложный единый подход Краммера–Зингера.

**One‑vs‑Rest (OvR)**, также известный как One‑vs‑All, заключается в обучении $K$ независимых бинарных классификаторов, каждый из которых отделяет объекты своего класса $k$ от объединения всех остальных классов. При предсказании новый объект $\mathbf{x}$ подаётся на вход всем $K$ моделям, и выбирается класс, для которого значение решающей функции $f_k(\mathbf{x})$ максимально:
$$
\hat{y} = \arg\max_{k \in \{1,\dots,K\}} f_k(\mathbf{x}).
$$
Преимущество OvR — относительно небольшое число моделей (ровно $K$). Недостаток — каждый классификатор обучается на сильно несбалансированной выборке, что может смещать границы в сторону малочисленного класса.

**One‑vs‑One (OvO)** строит $\binom{K}{2}$ бинарных классификаторов, по одному на каждую пару классов. При классификации каждый такой классификатор «голосует» за один из двух классов; побеждает класс, набравший наибольшее число голосов. Достоинство OvO — каждый классификатор работает на меньшей и сбалансированной выборке, что повышает надёжность обучения. Недостаток — квадратичный рост числа моделей, хотя для небольших $K$ это терпимо.

Метод **Краммера–Зингера** (Crammer–Singer) формулирует единую многоклассовую задачу с совместным обучением всех $K$ бинарных классификаторов, используя обобщённый hinge loss, аналогичный softmax для логистической регрессии, но без вероятностной нормировки. Этот подход реже используется на практике, уступая OvO по популярности.

Сравним стратегии OvO и OvR на примере распознавания цифр (`load_digits`):




In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
import time

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

for shape in ['ovo', 'ovr']:
    svm = SVC(kernel='rbf', decision_function_shape=shape)
    start = time.time()
    svm.fit(X_train, y_train)
    elapsed = time.time() - start
    y_pred = svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"{shape}: accuracy={acc:.3f}, macro F1={f1:.3f}, time={elapsed:.2f}s")


Результаты, как правило, близки, но OvO может давать чуть более высокое качество для данных с малым числом объектов на класс.

## 21. SVM для регрессии (SVR)

Идею максимизации зазора можно перенести и на задачи восстановления регрессии — получается метод **Support Vector Regression (SVR)**. Вместо того чтобы добиваться, чтобы все точки лежали вне полосы определённой ширины, в SVR требуется, чтобы как можно больше точек попадало **внутрь** симметричной трубки шириной $2\epsilon$ вокруг предсказывающей функции $f(\mathbf{x}) = \mathbf{w}^T\mathbf{x} + b$. Ошибки, не превышающие $\epsilon$, считаются несущественными и не штрафуются. Используется $\epsilon$**-нечувствительная функция потерь** (epsilon‑insensitive loss):
$$
L_\epsilon(y, f(\mathbf{x})) = \max(0, |y - f(\mathbf{x})| - \epsilon).
$$
Точки, выходящие за пределы трубки, штрафуются линейно с помощью slack‑переменных $\xi_i$ (для отклонений вверх) и $\xi_i^*$ (для отклонений вниз). Прямая задача SVR выглядит так:
$$
\boxed{\begin{aligned}
\min_{\mathbf{w},b,\boldsymbol{\xi},\boldsymbol{\xi}^*} \quad & \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} (\xi_i + \xi_i^*) \\
\text{s.t.} \quad & y_i - \mathbf{w}^T\mathbf{x}_i - b \le \epsilon + \xi_i, \\
& \mathbf{w}^T\mathbf{x}_i + b - y_i \le \epsilon + \xi_i^*, \\
& \xi_i \ge 0, \;\; \xi_i^* \ge 0, \quad i=1,\dots,n.
\end{aligned}}
\tag{21.1}
$$
Параметр $C$ управляет балансом между гладкостью решения (малым $\|\mathbf{w}\|$) и суммой выходов за трубку. При больших $\epsilon$ модель получается более простой и меньше чувствительной к шуму.

Двойственная задача SVR, как и в классификации, содержит только скалярные произведения, и к ней также применим ядровой трюк. Появляются два набора двойственных переменных $\alpha_i$ и $\alpha_i^*$, удовлетворяющих условиям $0 \le \alpha_i, \alpha_i^* \le C$ и $\sum (\alpha_i - \alpha_i^*) = 0$. Решающее правило:
$$
f(\mathbf{x}) = \sum_{i=1}^{n} (\alpha_i - \alpha_i^*) K(\mathbf{x}_i, \mathbf{x}) + b.
$$
Опорными векторами становятся объекты, для которых $\alpha_i - \alpha_i^* \neq 0$ — то есть точки, лежащие на границе трубки или вне её.

Продемонстрируем SVR на синтетических данных:




In [ ]:
from sklearn.svm import SVR
from sklearn.datasets import make_regression
import matplotlib.pyplot as plt
import numpy as np

X, y = make_regression(n_samples=100, n_features=1, noise=15.0, random_state=42)
svr = SVR(kernel='rbf', C=100, epsilon=10.0)
svr.fit(X, y)

X_plot = np.linspace(X.min(), X.max(), 200).reshape(-1,1)
y_pred = svr.predict(X_plot)

plt.figure(figsize=(8,5))
plt.scatter(X, y, alpha=0.6, label='данные')
plt.plot(X_plot, y_pred, 'r-', label='SVR')
plt.plot(X_plot, y_pred + svr.epsilon, 'r--', alpha=0.5)
plt.plot(X_plot, y_pred - svr.epsilon, 'r--', alpha=0.5)
plt.scatter(X[svr.support_], y[svr.support_], facecolors='none', edgecolors='k', s=100, label='опорные векторы')
plt.legend()
plt.title('SVR с $\epsilon$-трубкой')
plt.show()


График наглядно показывает трубку шириной $2\epsilon$ и опорные векторы, лежащие на её границах или вне их.

## 22. Теоретические основы: VC‑размерность и структурная минимизация риска

SVM имеет прочный теоретический фундамент в терминах теории вычислительного обучения. Ключевое понятие — **VC‑размерность** (Vapnik–Chervonenkis dimension) класса функций. Она измеряет способность семейства классификаторов «разбивать» произвольные наборы точек. Формально, VC‑размерность класса $\mathcal{F}$ есть максимальное число $m$, для которого существует набор из $m$ точек, который $\mathcal{F}$ может разбить всеми $2^m$ возможными способами.

Согласно фундаментальной теореме VC‑теории, с вероятностью не менее $1-\delta$ для любой функции $f \in \mathcal{F}$ истинный риск $R(f)$ ограничен сверху эмпирическим риском $R_{\text{emp}}(f)$ плюс штрафной член, зависящий от VC‑размерности $h$:
$$
R(f) \le R_{\text{emp}}(f) + \sqrt{\frac{h(\log(2n/h)+1) - \log(\delta/4)}{n}}.
$$
Таким образом, для минимизации ожидаемого риска нужно одновременно уменьшать эмпирический риск и сложность класса, измеряемую $h$. Это и есть принцип **структурной минимизации риска** (SRM), предложенный Вапником и Червоненкисом.

Для класса линейных классификаторов с зазором $\rho$ имеет место замечательная оценка. Предположим, что все обучающие векторы лежат внутри шара радиуса $R$. Тогда VC‑размерность $h$ класса гиперплоскостей, разделяющих данные с геометрическим зазором не менее $\rho$, удовлетворяет
$$
h \le \min\left(D, \left\lceil \frac{R^2}{\rho^2} \right\rceil\right) + 1,
$$
где $D$ — размерность пространства. Следовательно, максимизация зазора $\rho$ (эквивалентная минимизации $\|\mathbf{w}\|$) напрямую **уменьшает VC‑размерность**, а значит — и штрафной член в оценке обобщающей способности. Именно поэтому SVM, максимизируя зазор, находит решение, близкое к оптимальному по критерию SRM.

> **Углублённый взгляд: набросок доказательства оценки VC‑размерности через зазор**  
> Зафиксируем $n$ точек в шаре радиуса $R$. Покажем, что число различных дихотомий, которые может породить гиперплоскость с зазором не менее $\rho$, ограничено сверху величиной $\left(\frac{e n R}{\rho}\right)^{O(1/\rho^2)}$. Если это число меньше $2^n$, то не все $n$ точек могут быть разбиты произвольно, откуда и следует оценка VC‑размерности.  
> Основная идея: каждая разделяющая гиперплоскость с зазором $\rho$ однозначно определяется $O(1/\rho^2)$ точками на границе зазора. Рассматривая все возможные подмножества точек и перебирая эти «опорные» множества, можно подсчитать число допустимых гиперплоскостей. Комбинаторный перебор приводит к указанной верхней границе, из которой и извлекается асимптотика $h = O(R^2/\rho^2)$.

## 23. Сравнение SVM и логистической регрессии

Оба метода строят линейный классификатор, но опираются на разные функции потерь и дают разные сопутствующие свойства.

**Функции потерь.** SVM использует hinge loss $\max(0, 1 - y f)$, логистическая регрессия — log loss $\log(1 + \exp(-y f))$. Hinge loss обращается в ноль при $y f \ge 1$, т.е. для объектов, лежащих вне зазора, и не штрафует «излишнюю» уверенность. Log loss положителен для всех конечных отступов и неограниченно растёт при сильных ошибках, что делает его более чувствительным к выбросам. Обе функции являются выпуклыми мажорантами индикаторной потери $[y f \le 0]$.

**Интерпретируемость.** Логистическая регрессия даёт хорошо откалиброванные вероятности; SVM — только вещественный отступ. Чтобы получить вероятности от SVM, применяют **Platt scaling**: обучают дополнительную логистическую регрессию на значениях $f(\mathbf{x})$ (или на выходе `decision_function`) с валидационной выборкой. Это восстанавливает вероятностную шкалу, хотя и требует дополнительных усилий.

**Вычислительные аспекты.** Время обучения SVM растёт с числом опорных векторов, которое может достигать значительной доли от $n$. Логистическая регрессия обычно обучается быстрее на больших выборках, особенно с разреженными признаками, и не требует хранения ядерной матрицы.

**Качество.** SVM может превосходить логистическую регрессию, когда классы хорошо разделимы и выбрано подходящее ядро. LR предпочтительнее при большом числе признаков (например, в текстовой классификации), для разреженных данных и когда важна вероятностная интерпретация.

Сведём ключевые различия в таблицу.

| Критерий | Логистическая регрессия | SVM |
|----------|--------------------------|-----|
| Функция потерь | Log loss | Hinge loss |
| Выход модели | Вероятность $p(y=1 \mid \mathbf{x})$ | Отступ $f(\mathbf{x})$ |
| Вероятностная калибровка | Встроенная | Требует Platt scaling |
| Чувствительность к выбросам | Более чувствительна (log loss не насыщается) | Менее чувствительна (hinge loss обнуляется для «лёгких» примеров) |
| Ядра | Неявно, через ядерную логистическую регрессию | Естественная поддержка ядрового трюка |
| Разреженность решения | Использует все объекты | Использует только опорные векторы |
| Скорость обучения | Быстрая, особенно с SGD | Медленнее при больших $n$, зависит от числа опорных векторов |
| Интерпретируемость | Высокая (отношение шансов) | Средняя (опорные векторы, но нет вероятностей) |

## 24. Мост к нейронным сетям и kernel methods

SVM с RBF‑ядром можно интерпретировать как нейронную сеть с одним скрытым слоем. Действительно, решающее правило $f(\mathbf{x}) = \sum_{i} \alpha_i y_i \exp(-\gamma \|\mathbf{x} - \mathbf{x}_i\|^2) + b$ представляет собой линейную комбинацию радиальных базисных функций, центры которых — опорные векторы, а выходной нейрон формирует знак взвешенной суммы. Эта связь исторически способствовала переносу идей ядерных методов в нейронные сети и наоборот.

Главное ограничение ядерного SVM — число опорных векторов обычно растёт линейно с размером выборки, что делает предсказание медленным при больших $n$. Кроме того, выбор ядра и его параметров требует тщательной настройки. Глубокие нейронные сети, напротив, учат иерархическое представление признаков автоматически из данных, избегая фиксированного ядра, и за счёт мини‑батчевого обучения хорошо масштабируются на огромные выборки. Тем не менее, для многих задач с умеренным количеством данных SVM остаётся надёжным и теоретически обоснованным инструментом.



## 25. Заключение и резюме главы

В этой главе мы проследили путь от простейшей геометрической интуиции о разделяющей гиперплоскости до сложного, но элегантного семейства методов опорных векторов. Ключевые концепции, которые следует вынести:

- **Зазор** (margin) — мера устойчивости разделяющей гиперплоскости; максимизация зазора ведёт к лучшей обобщающей способности.
- **Опорные векторы** — разреженное множество точек, полностью определяющих решение; именно на них «опирается» гиперплоскость.
- **Двойственность** позволяет перейти от прямой задачи минимизации нормы к двойственной задаче, зависящей только от скалярных произведений.
- **Ядровой трюк** неявно отображает данные в спрямляющее пространство высокой размерности, не требуя явного вычисления отображения.
- **Алгоритм SMO** даёт эффективный способ обучения SVM путём последовательной оптимизации пар множителей.
- **Структурная минимизация риска** связывает максимизацию зазора с уменьшением VC‑размерности, давая гарантии обобщения.

Итоговые сопоставительные таблицы обобщают основные варианты SVM и сравнение с другими методами.

**Таблица: Hard‑margin vs Soft‑margin vs SVR**

| Характеристика | Hard‑margin | Soft‑margin | SVR |
|----------------|------------|-------------|-----|
| Цель | Идеальное разделение классов | Разделение с допущением ошибок | Аппроксимация непрерывной функции |
| Дополнительные переменные | Нет | $\xi_i \ge 0$ | $\xi_i, \xi_i^* \ge 0$ |
| Параметр компромисса | Нет | $C$ — вес ошибок | $C$ — вес ошибок, $\epsilon$ — ширина трубки |
| Ограничения | $y_i f(\mathbf{x}_i) \ge 1$ | $y_i f(\mathbf{x}_i) \ge 1 - \xi_i$ | $\mid y_i - f(\mathbf{x}_i) \mid \le \epsilon + \xi_i^{(*)}$ |
| Применимость | Только линейно разделимые данные | Любые данные | Произвольная регрессия |

**Таблица: SVM vs Logistic Regression (повторена для удобства заключения)**  
— приведена выше.



## 26. Вопросы для самопроверки (объединяют всю главу)

### Для начинающих

1. Что именно максимизирует SVM при построении разделяющей гиперплоскости? Почему эта стратегия предпочтительнее простого разделения классов?
2. Какую роль в soft‑margin SVM играет параметр $C$? Что происходит при $C \to 0$ и $C \to \infty$?
3. Зачем в SVM используются ядра? Приведите пример нелинейных данных, для которых линейный SVM неэффективен.
4. В чём отличие стратегий One‑vs‑One и One‑vs‑Rest для многоклассовой классификации? Какая из них обучает больше моделей?
5. Чем задача SVR отличается от обычной линейной регрессии? Что такое $\epsilon$‑трубка?
6. Почему SVM не выдаёт вероятности классов напрямую, и как их можно получить?

### Для профессионалов

1. Объясните, как максимизация геометрического зазора в SVM связана с минимизацией VC‑размерности и реализует принцип структурной минимизации риска.
2. Приведите набросок доказательства оценки VC‑размерности линейных классификаторов с зазором $\rho$ через радиус охватывающего шара $R$.
3. Выведите двойственную задачу для SVR (прямая задача приведена в (21.1)). Какие ограничения возникают на двойственные переменные?
4. Опишите процедуру Platt scaling для калибровки вероятностей SVM. Какие данные используются для обучения калибровочной модели?
5. Сравните hinge loss и log loss с точки зрения робастности к выбросам и влияния на разреженность решения. Как эти свойства проявляются в поведении SVM и логистической регрессии?

## 27. Задания повышенной сложности

1. **Двойственная задача SVR.** Выведите двойственную задачу для SVR, начиная с лагранжиана и условий KKT. Покажите, что решение выражается через разность $\alpha_i - \alpha_i^*$, и поясните, какие точки становятся опорными векторами.
2. **Platt scaling.** Реализуйте Platt scaling для вашего класса `SimpleSVM` из предыдущей части. Разбейте данные на обучение и валидацию, обучите SVM, затем логистическую регрессию на выходах решающей функции валидационного набора. Постройте калибровочные кривые до и после калибровки.
3. **Кривые обучения.** Проведите эксперимент: для синтетического набора данных постройте графики accuracy на обучении и на тесте в зависимости от размера обучающей выборки для SVM с RBF‑ядром и для логистической регрессии. Проанализируйте компромисс смещения и дисперсии (bias‑variance trade‑off) для обеих моделей.
4. **Единственность hard‑margin решения.** Дайте строгое доказательство того, что в задаче hard‑margin SVM вектор $\mathbf{w}^*$ единственен, а смещение $b^*$ может быть неединственным только в вырожденных случаях (например, когда все опорные векторы лежат на одинаковом расстоянии от гиперплоскости).
5. **Выражение зазора через опорные векторы.** Используя формулы (3.2) и (5.1), покажите, что геометрический зазор $\rho = 2 / \|\mathbf{w}\|$, и выразите $\|\mathbf{w}\|^2$ через двойственные переменные: $\|\mathbf{w}\|^2 = \sum_{i,j} \alpha_i \alpha_j y_i y_j K(\mathbf{x}_i, \mathbf{x}_j)$. Докажите, что все свободные опорные векторы ($0 < \alpha_i < C$) дают одно и то же значение $b$.